In [1]:
# ============================================================
# 025_end_to_end_orchestration_demo
# ============================================================
#
# Overview
# ----------------
# Day25 stage-gate demo: orchestrate the full research automation pipeline
# from seed corpus discovery through daily scanning and RQ updates, producing
# weekly and daily summary outputs. Executes seven prior notebooks (018-024)
# in sequence, compiles results into five deliverable reports, and implements
# a minimal CLI/GitHub Actions trigger skeleton with failure alerting.
#
# Inputs / Outputs
# ----------------
# Inputs:
#   - Environment variables from env.txt (API keys, Notion config)
#   - Existing notebooks/modules 018-024 (called as functions or via papermill)
#   - Notion databases: LIT_DB, RQ_DB, MTG_DB
#   - Google Drive folder for PDFs
#
# Outputs:
#   - runs/YYYY-MM-DD_HHMM/weekly_corpus_update_report.md
#   - runs/YYYY-MM-DD_HHMM/network_update_summary.md
#   - runs/YYYY-MM-DD_HHMM/gap_update.md
#   - runs/YYYY-MM-DD_HHMM/rq_update_proposal.md
#   - runs/YYYY-MM-DD_HHMM/x_draft.md
#   - runs/YYYY-MM-DD_HHMM/run_manifest.json
#   - artifacts/ (intermediate checkpoint files)
#   - Notion summaries (optional, non-destructive)
#
# Structure
# ----------------
# Cell 01: Environment setup and dependency imports
# Cell 02: Load and validate environment variables and API credentials
# Cell 03: Notion schema definitions and configuration
# Cell 04: Initialize OpenAI client and validate Notion authentication
# Cell 05: Define orchestration config dataclass and run directory setup
# Cell 06: Implement step function placeholders for notebooks 018-024
# Cell 07: Implement failure handling and notification stubs
# Cell 08: Define report compilation functions for five output documents
# Cell 09: Implement main pipeline orchestrator function
# Cell 10: Execute weekly corpus update steps (018-020)
# Cell 11: Execute network and gap analysis steps (021-022)
# Cell 12: Execute daily scanning and RQ update steps (023-024)
# Cell 13: Compile final reports and write to run directory
# Cell 14: Generate run manifest and summary statistics
# Cell 15: CLI and GitHub Actions orchestration skeleton
# Cell 16: Run the full end-to-end pipeline with dry-run option
#
# Notes
# ----------------
# - Uses placeholder step functions with TODO markers for actual notebook integration
# - Supports dry-run mode to prevent writes to Notion/Drive during testing
# - All outputs timestamped to runs/YYYY-MM-DD_HHMM/ for deterministic tracking
# - Implements robust error handling with step-level failure capture
# - Includes minimal stubs for Notion ticket creation and notification alerts
# - Can be extended to use papermill/nbclient for actual notebook execution
# - Safe, non-destructive Notion writes (append-only summaries)

In [2]:
# ============================================================
# Cell 01 — Environment setup and dependency imports
# ============================================================
# Overview:
#   Import all dependencies required for end-to-end orchestration:
#   - Standard library (os, json, datetime, pathlib, dataclasses)
#   - External packages (openai, notion_client, papermill/nbclient stubs)
#   - Logging and error handling utilities
#
# Inputs / Outputs:
#   Inputs: None
#   Outputs: Configured runtime environment, imported modules
#
# Notes:
#   - All imports placed here for clear dependency tracking
#   - Logging configured to both console and file
#   - Supports dry-run mode flag for testing
#

# --- Mandatory env loading ---
from dotenv import load_dotenv
load_dotenv('env.txt')

# --- Runtime LLM configuration (given / assumed) ---
llm_provider = 'OpenAI'
llm_model = 'gpt-4o-mini'
llm_temperature = 0.0

# --- Standard library imports ---
import os
import json
import logging
from datetime import datetime
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Dict, List, Optional, Any, Tuple
import traceback
import sys

# --- External dependencies ---
try:
    from openai import OpenAI
except ImportError:
    OpenAI = None  # Will validate in cell 04

try:
    from notion_client import Client as NotionClient
except ImportError:
    NotionClient = None  # Will validate in cell 04

# TODO: Uncomment when using papermill for notebook execution
# try:
#     import papermill as pm
# except ImportError:
#     pm = None

# TODO: Uncomment when using nbclient for notebook execution
# try:
#     from nbclient import NotebookClient
# except ImportError:
#     NotebookClient = None

# --- Logging configuration ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.StreamHandler(sys.stdout)
    ]
)

logger = logging.getLogger('orchestrator')
logger.info(f"Environment loaded. LLM: {llm_provider}/{llm_model} @ temp={llm_temperature}")

# --- Global configuration flags ---
DRY_RUN = os.getenv('DRY_RUN', 'false').lower() == 'true'
VERBOSE = os.getenv('VERBOSE', 'false').lower() == 'true'

if DRY_RUN:
    logger.warning("🔶 DRY_RUN mode enabled - no writes to Notion/Drive will be performed")
if VERBOSE:
    logger.setLevel(logging.DEBUG)
    logger.debug("Verbose logging enabled")


2026-01-20 10:28:39 [INFO] orchestrator: Environment loaded. LLM: OpenAI/gpt-4o-mini @ temp=0.0


In [4]:
# ============================================================
# Cell 02 — Load and validate environment variables and API credentials
# ============================================================
# Overview:
#   Load all required environment variables from env.txt and validate
#   that essential credentials (OpenAI API key, Notion tokens, database IDs)
#   are present. Uses explicit variable-to-attribute mapping to avoid
#   silent misconfiguration.
#
# Inputs / Outputs:
#   Inputs: env.txt file with API keys and configuration
#   Outputs: env_config object with validated values and aliases
#
# Notes:
#   - Does NOT initialize API clients (auth check is done in later cells)
#   - Uses NOTION_*_DB_ID as the source of truth
#   - Provides backward-compatible aliases: lit_db / rq_db / mtg_db
#

from typing import Optional, List, Dict, Any
import os
import json

logger.info("Loading environment configuration from env.txt...")

# ------------------------------------------------------------
# Required environment variables (single source of truth)
# ------------------------------------------------------------
REQUIRED_ENV_VARS = {
    "OPENAI_API_KEY": "openai_api_key",
    "NOTION_TOKEN": "notion_token",
    "NOTION_VERSION": "notion_version",
    "NOTION_LIT_DB_ID": "notion_lit_db_id",
    "NOTION_RQ_DB_ID": "notion_rq_db_id",
    "NOTION_MTG_DB_ID": "notion_mtg_db_id",
}

# ------------------------------------------------------------
# Optional environment variables with defaults
# ------------------------------------------------------------
OPTIONAL_ENV_VARS = {
    "GOOGLE_DRIVE_FOLDER_ID": None,
    "ARXIV_SEARCH_MAX_RESULTS": "50",
    "SEMANTIC_SCHOLAR_SEARCH_MAX_RESULTS": "50",
    "CORPUS_UPDATE_FREQUENCY": "weekly",  # 'weekly' or 'daily'
    "MAX_CONCURRENT_API_CALLS": "5",
    "TIMEOUT_SECONDS": "300",
    "NOTIFICATION_EMAIL": None,
    "SLACK_WEBHOOK_URL": None,
    "GITHUB_TOKEN": None,
}

# ------------------------------------------------------------
# Environment configuration container
# ------------------------------------------------------------
class EnvConfig:
    def __init__(self):
        self.errors: List[str] = []
        self.warnings: List[str] = []

        # Required
        self.openai_api_key: Optional[str] = None
        self.notion_token: Optional[str] = None
        self.notion_version: Optional[str] = None
        self.notion_lit_db_id: Optional[str] = None
        self.notion_rq_db_id: Optional[str] = None
        self.notion_mtg_db_id: Optional[str] = None

        # Backward-compatible aliases (used by orchestration code)
        self.lit_db: Optional[str] = None
        self.rq_db: Optional[str] = None
        self.mtg_db: Optional[str] = None

        # Optional
        self.google_drive_folder_id: Optional[str] = None
        self.arxiv_search_max_results: int = 50
        self.semantic_scholar_search_max_results: int = 50
        self.corpus_update_frequency: str = "weekly"
        self.max_concurrent_api_calls: int = 5
        self.timeout_seconds: int = 300
        self.notification_email: Optional[str] = None
        self.slack_webhook_url: Optional[str] = None
        self.github_token: Optional[str] = None

    def validate(self) -> bool:
        # --- Required vars ---
        for env_var, attr in REQUIRED_ENV_VARS.items():
            value = os.getenv(env_var)
            if not value or value.strip() == "":
                self.errors.append(f"Missing required environment variable: {env_var}")
            else:
                setattr(self, attr, value.strip())

        # --- Aliases ---
        self.lit_db = self.notion_lit_db_id
        self.rq_db = self.notion_rq_db_id
        self.mtg_db = self.notion_mtg_db_id

        # --- Optional vars ---
        for env_var, default in OPTIONAL_ENV_VARS.items():
            value = os.getenv(env_var, default)

            if env_var in ["ARXIV_SEARCH_MAX_RESULTS", "SEMANTIC_SCHOLAR_SEARCH_MAX_RESULTS"]:
                try:
                    setattr(self, env_var.lower(), int(value))
                except Exception:
                    self.warnings.append(
                        f"Invalid integer for {env_var}, using default {default}"
                    )
                    setattr(self, env_var.lower(), int(default))
            elif env_var in ["MAX_CONCURRENT_API_CALLS", "TIMEOUT_SECONDS"]:
                try:
                    setattr(self, env_var.lower(), int(value))
                except Exception:
                    self.warnings.append(
                        f"Invalid integer for {env_var}, using default {default}"
                    )
                    setattr(self, env_var.lower(), int(default))
            else:
                setattr(self, env_var.lower(), value)

        # --- Validation rules ---
        if self.corpus_update_frequency not in ["weekly", "daily"]:
            self.warnings.append(
                f"CORPUS_UPDATE_FREQUENCY must be 'weekly' or 'daily', got '{self.corpus_update_frequency}'. Using 'weekly'."
            )
            self.corpus_update_frequency = "weekly"

        if not self.google_drive_folder_id:
            self.warnings.append("GOOGLE_DRIVE_FOLDER_ID not set - Drive PDF ingestion disabled")

        if not self.notification_email and not self.slack_webhook_url:
            self.warnings.append("No notification endpoint configured (EMAIL or SLACK)")

        return len(self.errors) == 0

    def to_dict(self) -> Dict[str, Any]:
        """Non-sensitive snapshot for logs / manifests."""
        return {
            "lit_db": self.lit_db,
            "rq_db": self.rq_db,
            "mtg_db": self.mtg_db,
            "corpus_update_frequency": self.corpus_update_frequency,
            "arxiv_search_max_results": self.arxiv_search_max_results,
            "semantic_scholar_search_max_results": self.semantic_scholar_search_max_results,
            "max_concurrent_api_calls": self.max_concurrent_api_calls,
            "timeout_seconds": self.timeout_seconds,
            "has_openai_key": bool(self.openai_api_key),
            "has_notion_token": bool(self.notion_token),
            "has_notifications": bool(self.notification_email or self.slack_webhook_url),
        }

# ------------------------------------------------------------
# Load & validate
# ------------------------------------------------------------
env_config = EnvConfig()

if not env_config.validate():
    logger.error("❌ Environment validation failed:")
    for error in env_config.errors:
        logger.error(f"  - {error}")
    raise RuntimeError(
        f"Missing {len(env_config.errors)} required environment variable(s). "
        "Please check env.txt and ensure all required variables are set."
    )

if env_config.warnings:
    logger.warning("⚠️  Environment validation warnings:")
    for warning in env_config.warnings:
        logger.warning(f"  - {warning}")

logger.info("✅ Environment validation successful")
logger.info(f"   LIT_DB: {env_config.lit_db[:8]}...")
logger.info(f"   RQ_DB:  {env_config.rq_db[:8]}...")
logger.info(f"   MTG_DB: {env_config.mtg_db[:8]}...")
logger.info(f"   Corpus update frequency: {env_config.corpus_update_frequency}")

if VERBOSE:
    logger.debug("Environment configuration snapshot:")
    logger.debug(json.dumps(env_config.to_dict(), indent=2))


2026-01-20 10:31:38 [INFO] orchestrator: Loading environment configuration from env.txt...
2026-01-20 10:31:38 [WARNING] orchestrator: ⚠️  Environment validation warnings:
2026-01-20 10:31:38 [WARNING] orchestrator:   - GOOGLE_DRIVE_FOLDER_ID not set - Drive PDF ingestion disabled
2026-01-20 10:31:38 [WARNING] orchestrator:   - No notification endpoint configured (EMAIL or SLACK)
2026-01-20 10:31:38 [INFO] orchestrator: ✅ Environment validation successful
2026-01-20 10:31:38 [INFO] orchestrator:    LIT_DB: 2a98e0e4...
2026-01-20 10:31:38 [INFO] orchestrator:    RQ_DB:  2a98e0e4...
2026-01-20 10:31:38 [INFO] orchestrator:    MTG_DB: 2a98e0e4...
2026-01-20 10:31:38 [INFO] orchestrator:    Corpus update frequency: weekly


In [8]:
# ============================================================
# Cell 03 — Notion schema definitions and configuration
# ============================================================
# Overview:
#   Declare canonical schema mappings for Notion databases (Literature, RQ, Meetings),
#   and provide safe property extraction helpers that always reference the mappings.
#   This keeps the pipeline consistent with the existing Notion DB property names.
#
# Inputs / Outputs:
#   Inputs: Database IDs from env_config (validated in cell 02)
#   Outputs: SCHEMA_CONFIG + helper functions for safe extraction
#
# Notes:
#   - This notebook uses LIT_SCHEMA / MEETING_SCHEMA / RQ_SCHEMA as the single source of truth.
#   - Avoids hard-coded property names like 'Title'/'Question'/'Meeting Name' (they drift easily).
#   - Does not modify schemas; assumes they exist in Notion.
#

from typing import Optional, List, Dict, Any

# --- Notion Literature DB schema mapping ---
# Property names for querying and extracting paper data
LIT_SCHEMA = {
    "title": "Name",                 # title
    "created_time": "Created time",  # created_time 
    "authors_year": "Authors & Year",
    "tags": "Tags",
    "pdf_link": "PDF Link",
    "findings": "Findings",
    "core_idea": "Core Idea",
    "notes": "Notes",
    "methods": "Methods",
    "type": "Type",
    "source": "Source",
    "datasets": "Datasets",
    "papers_rel": "Papers",          # relation
}

# --- Notion Meeting DB schema mapping ---
MEETING_SCHEMA = {
    "title": "Name",
    "date": "Date",
    "tags": "Tags",
    "rq_mentions": "Papers",       # ここは “RQ mentions” ではなく、関連論文 relation
    "notes_primary": "Summary",    # まずSummaryを優先
    "notes_fallback": "Transcript",
    "interviewee": "Interviewee",
}


# --- Notion RQ DB schema mapping ---
RQ_SCHEMA = {
    "title": "Name",
    "status": "Status",
    "priority": "Priority",
    "tags": "Tags",
    "evidence": "Linked Paper",
    "rationale": "Rationale / Background",
    "approach": "Proposed Approach",
    "gap": "Gap Identified",
}

# ------------------------------------------------------------
# Schema registry (single source of truth)
# ------------------------------------------------------------
SCHEMA_CONFIG = {
    "lit_db": {
        "database_id": env_config.lit_db,
        "schema": LIT_SCHEMA,  # mapping keys -> Notion property names
        "name": "Literature Database",
    },
    "rq_db": {
        "database_id": env_config.rq_db,
        "schema": RQ_SCHEMA,
        "name": "Research Questions Database",
    },
    "mtg_db": {
        "database_id": env_config.mtg_db,
        "schema": MEETING_SCHEMA,
        "name": "Meetings Database",
    },
}

# ------------------------------------------------------------
# Generic property helpers (mapping-driven)
# ------------------------------------------------------------
def _get_prop(page: Dict[str, Any], notion_prop_name: str) -> Dict[str, Any]:
    return page.get("properties", {}).get(notion_prop_name, {}) or {}

def extract_title_from_mapping(page: Dict[str, Any], title_prop_name: str) -> Optional[str]:
    """Extract Notion title property."""
    try:
        prop = _get_prop(page, title_prop_name)
        if "title" in prop:
            parts = prop.get("title") or []
            return "".join([p.get("plain_text", "") for p in parts]).strip() or None
    except Exception as e:
        logger.debug(f"Error extracting title '{title_prop_name}': {e}")
    return None

def extract_rich_text_from_mapping(page: Dict[str, Any], prop_name: str) -> Optional[str]:
    """Extract Notion rich_text property."""
    try:
        prop = _get_prop(page, prop_name)
        if "rich_text" in prop:
            parts = prop.get("rich_text") or []
            text = "".join([p.get("plain_text", "") for p in parts]).strip()
            return text or None
    except Exception as e:
        logger.debug(f"Error extracting rich_text '{prop_name}': {e}")
    return None

def extract_select_from_mapping(page: Dict[str, Any], prop_name: str) -> Optional[str]:
    """Extract Notion select property."""
    try:
        prop = _get_prop(page, prop_name)
        sel = prop.get("select")
        if sel:
            return sel.get("name")
    except Exception as e:
        logger.debug(f"Error extracting select '{prop_name}': {e}")
    return None

def extract_multi_select_from_mapping(page: Dict[str, Any], prop_name: str) -> List[str]:
    """Extract Notion multi_select property."""
    try:
        prop = _get_prop(page, prop_name)
        items = prop.get("multi_select") or []
        return [i.get("name", "") for i in items if i.get("name")]
    except Exception as e:
        logger.debug(f"Error extracting multi_select '{prop_name}': {e}")
    return []

def extract_number_from_mapping(page: Dict[str, Any], prop_name: str) -> Optional[float]:
    """Extract Notion number property."""
    try:
        prop = _get_prop(page, prop_name)
        if "number" in prop:
            return prop.get("number")
    except Exception as e:
        logger.debug(f"Error extracting number '{prop_name}': {e}")
    return None

def extract_url_from_mapping(page: Dict[str, Any], prop_name: str) -> Optional[str]:
    """Extract Notion url property."""
    try:
        prop = _get_prop(page, prop_name)
        if "url" in prop:
            return prop.get("url")
    except Exception as e:
        logger.debug(f"Error extracting url '{prop_name}': {e}")
    return None

def extract_date_from_mapping(page: Dict[str, Any], prop_name: str) -> Optional[str]:
    """Extract Notion date property (ISO start)."""
    try:
        prop = _get_prop(page, prop_name)
        date_obj = prop.get("date")
        if date_obj:
            return date_obj.get("start")
    except Exception as e:
        logger.debug(f"Error extracting date '{prop_name}': {e}")
    return None

def extract_relation_ids_from_mapping(page: Dict[str, Any], prop_name: str) -> List[str]:
    """Extract Notion relation IDs."""
    try:
        prop = _get_prop(page, prop_name)
        rels = prop.get("relation") or []
        return [r.get("id", "") for r in rels if r.get("id")]
    except Exception as e:
        logger.debug(f"Error extracting relation '{prop_name}': {e}")
    return []

# ------------------------------------------------------------
# Convenience wrappers per DB (so downstream code is readable)
# ------------------------------------------------------------
def lit_title(page: Dict[str, Any]) -> Optional[str]:
    return extract_title_from_mapping(page, LIT_SCHEMA["title"])  # e.g., "Name"

def rq_title(page: Dict[str, Any]) -> Optional[str]:
    return extract_title_from_mapping(page, RQ_SCHEMA["title"])   # e.g., "Name"

def mtg_title(page: Dict[str, Any]) -> Optional[str]:
    return extract_title_from_mapping(page, MEETING_SCHEMA["title"])  # e.g., "Name"

logger.info("✅ Notion schema mappings loaded (mapping-driven)")
logger.info(f"   LIT_DB: {SCHEMA_CONFIG['lit_db']['database_id'][:8]}...")
logger.info(f"   RQ_DB:  {SCHEMA_CONFIG['rq_db']['database_id'][:8]}...")
logger.info(f"   MTG_DB: {SCHEMA_CONFIG['mtg_db']['database_id'][:8]}...")

if VERBOSE:
    logger.debug("Schema mappings (keys -> Notion property names):")
    logger.debug(json.dumps({
        "LIT_SCHEMA": LIT_SCHEMA,
        "RQ_SCHEMA": RQ_SCHEMA,
        "MEETING_SCHEMA": MEETING_SCHEMA,
    }, indent=2, ensure_ascii=False))


2026-01-20 10:35:42 [INFO] orchestrator: ✅ Notion schema mappings loaded (mapping-driven)
2026-01-20 10:35:42 [INFO] orchestrator:    LIT_DB: 2a98e0e4...
2026-01-20 10:35:42 [INFO] orchestrator:    RQ_DB:  2a98e0e4...
2026-01-20 10:35:42 [INFO] orchestrator:    MTG_DB: 2a98e0e4...


In [13]:
# ============================================================
# Cell 04 — Initialize OpenAI + Notion (REST) and validate DB schemas + manifest
# ============================================================
# Overview:
#   - Initialize OpenAI client and run a minimal auth test.
#   - Initialize Notion REST client (requests-based) and validate auth.
#   - Retrieve each Notion database via REST and validate:
#       (1) DB is accessible
#       (2) DB properties are non-empty (sanity)
#       (3) All expected Notion property names exist (mapping-driven validation)
#   - Emit validation_summary and optionally persist to run_dir.
#
# Inputs / Outputs:
#   Inputs:
#     - env_config (Cell 02)
#     - SCHEMA_CONFIG with dict mappings (Cell 03): {db_key: {database_id, schema(dict), name}}
#     - llm_model, llm_temperature, VERBOSE
#     - optional: run_dir (Path)
#   Outputs:
#     - openai_client
#     - notion_rest (requests wrapper)
#     - validation_summary (dict)
#
# Notes:
#   - Uses REST as the source of truth because notion-client SDK may omit "properties".
#   - Does real API calls (not dry-run safe).
#   - Does not print secrets.
#

from datetime import datetime
from typing import Dict, Any, List, Optional
import json
import requests

# ---------------------------------------------------------------------
# 0) Build Notion REST headers (self-contained)
# ---------------------------------------------------------------------
NOTION_HEADERS = {
    "Authorization": f"Bearer {env_config.notion_token}",
    "Notion-Version": env_config.notion_version,
    "Content-Type": "application/json",
}

# ---------------------------------------------------------------------
# 1) OpenAI init + minimal auth test
# ---------------------------------------------------------------------
logger.info("Initializing OpenAI client...")

if OpenAI is None:
    raise ImportError("OpenAI package not installed. Install with: pip install openai")

try:
    openai_client = OpenAI(api_key=env_config.openai_api_key)

    logger.debug("Testing OpenAI authentication with minimal completion...")
    _test_response = openai_client.chat.completions.create(
        model=llm_model,
        messages=[{"role": "user", "content": "ping"}],
        max_tokens=5,
        temperature=float(llm_temperature) if llm_temperature is not None else 0.0,
    )

    if not _test_response or not getattr(_test_response, "choices", None):
        raise ValueError("OpenAI test call returned empty response")

    logger.info("✅ OpenAI client authenticated successfully")
    logger.info(f"   Model: {llm_model}")
    logger.info(f"   Temperature: {llm_temperature}")

except Exception as e:
    logger.error(f"❌ OpenAI authentication failed: {e}")
    raise RuntimeError(
        "Failed to authenticate OpenAI client. "
        "Please verify OPENAI_API_KEY in env.txt."
        f" Error: {e}"
    )

# ---------------------------------------------------------------------
# 2) Notion REST client (requests-based)
# ---------------------------------------------------------------------
class NotionRESTClient:
    def __init__(self, headers: Dict[str, str]):
        self.headers = headers

    def _check(self, r: requests.Response, context: str) -> Dict[str, Any]:
        if r.status_code >= 400:
            try:
                detail = r.json()
            except Exception:
                detail = {"text": r.text}
            raise RuntimeError(f"Notion API error ({context}): HTTP {r.status_code} - {detail}")
        return r.json()

    def users_me(self) -> Dict[str, Any]:
        r = requests.get("https://api.notion.com/v1/users/me", headers=self.headers, timeout=30)
        return self._check(r, "users.me")

    def retrieve_database(self, database_id: str) -> Dict[str, Any]:
        url = f"https://api.notion.com/v1/databases/{database_id}"
        r = requests.get(url, headers=self.headers, timeout=30)
        return self._check(r, f"databases.retrieve:{database_id}")

    def query_database(self, database_id: str, payload: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
        url = f"https://api.notion.com/v1/databases/{database_id}/query"
        r = requests.post(url, headers=self.headers, json=payload or {}, timeout=30)
        return self._check(r, f"databases.query:{database_id}")

notion_rest = NotionRESTClient(headers=NOTION_HEADERS)

logger.info("Initializing Notion REST client...")
try:
    user_info = notion_rest.users_me()
    logger.info("✅ Notion authenticated successfully (REST)")
    logger.info(f"   Bot name: {user_info.get('name', 'Unknown')}")
    logger.info(f"   Bot type: {user_info.get('type', 'Unknown')}")
except Exception as e:
    logger.error(f"❌ Notion authentication failed (REST): {e}")
    raise RuntimeError(
        "Failed to authenticate Notion via REST. "
        "Please verify NOTION_TOKEN and NOTION_VERSION in env.txt."
        f" Error: {e}"
    )

# ---------------------------------------------------------------------
# 3) DB validation (REST source of truth)
# ---------------------------------------------------------------------
logger.info("Validating Notion database access (REST)...")

db_validation_results: Dict[str, Dict[str, Any]] = {}
failed_dbs: List[str] = []

for db_key, cfg in SCHEMA_CONFIG.items():
    db_id = cfg["database_id"]
    db_label = cfg["name"]
    schema_mapping = cfg["schema"]  # dict: canonical_key -> Notion property name

    try:
        db_info = notion_rest.retrieve_database(db_id)

        # title
        title_parts = db_info.get("title", []) or []
        title = "".join([t.get("plain_text", "") for t in title_parts]).strip() or "Untitled"

        # properties
        properties = db_info.get("properties", {}) or {}
        prop_count = len(properties)

        # sanity check
        sanity_ok = prop_count > 0
        sanity_warning = None
        if not sanity_ok:
            sanity_warning = (
                "Database properties returned as empty. This is unusual. "
                "Double-check the database ID and integration access."
            )

        # mapping-driven validation
        expected_props = list(schema_mapping.values())
        missing_props = [p for p in expected_props if p not in properties]

        db_validation_results[db_key] = {
            "success": True,
            "database_id": db_id,
            "label": db_label,
            "title": title,
            "property_count": prop_count,
            "sanity_ok": sanity_ok,
            "sanity_warning": sanity_warning,
            "expected_property_count": len(expected_props),
            "missing_properties": missing_props,
        }

        logger.info(f"   ✅ {db_label}: '{title}' ({prop_count} properties)")

        if sanity_warning:
            logger.warning(f"      ⚠️  Sanity check: {sanity_warning}")

        if missing_props:
            logger.warning(
                f"      ⚠️  Missing {len(missing_props)} expected Notion properties: {missing_props[:8]}..."
            )
        else:
            logger.debug(f"      All {len(expected_props)} expected properties present")

        if VERBOSE:
            logger.debug(f"      Expected properties (first 15): {expected_props[:15]}")
            logger.debug(f"      Retrieved property keys (first 25): {list(properties.keys())[:25]}")

    except Exception as e:
        logger.error(f"   ❌ {db_label}: failed to retrieve/validate - {e}")
        db_validation_results[db_key] = {
            "success": False,
            "database_id": db_id,
            "label": db_label,
            "error": str(e),
        }
        failed_dbs.append(db_key)

# hard fail only if database retrieval fails
if failed_dbs:
    logger.error("❌ Database retrieval/validation failed for:")
    for k in failed_dbs:
        logger.error(f"   - {SCHEMA_CONFIG[k]['name']}: {db_validation_results[k].get('error')}")
    raise RuntimeError(
        f"Failed to access {len(failed_dbs)} Notion database(s). "
        "Please verify database IDs and integration permissions."
    )

logger.info("✅ All Notion databases retrieved successfully (REST)")

# ---------------------------------------------------------------------
# 4) validation manifest output
# ---------------------------------------------------------------------
validation_summary = {
    "timestamp": datetime.now().isoformat(),
    "openai": {
        "model": llm_model,
        "temperature": llm_temperature,
        "authenticated": True,
    },
    "notion": {
        "authenticated": True,
        "bot_name": user_info.get("name", "Unknown"),
        "bot_type": user_info.get("type", "Unknown"),
        "version": env_config.notion_version,
        "transport": "REST(requests)",
    },
    "databases": db_validation_results,
}

# Save manifest if run_dir exists
try:
    if "run_dir" in globals() and run_dir is not None:
        manifest_path = run_dir / "validation_summary.json"
        manifest_path.write_text(
            json.dumps(validation_summary, indent=2, ensure_ascii=False, default=str),
            encoding="utf-8",
        )
        logger.info(f"✅ Validation manifest saved: {manifest_path}")
    else:
        logger.info("ℹ️  run_dir not set; validation manifest not saved to disk (available as `validation_summary`).")
except Exception as e:
    logger.warning(f"⚠️  Failed to write validation manifest: {e}")

if VERBOSE:
    logger.debug(json.dumps(validation_summary, indent=2, ensure_ascii=False, default=str))

logger.info("Client initialization + Notion DB validation (REST) complete. Ready for orchestration.")


2026-01-20 10:41:32 [INFO] orchestrator: Initializing OpenAI client...
2026-01-20 10:41:33 [INFO] httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-20 10:41:33 [INFO] orchestrator: ✅ OpenAI client authenticated successfully
2026-01-20 10:41:33 [INFO] orchestrator:    Model: gpt-4o-mini
2026-01-20 10:41:33 [INFO] orchestrator:    Temperature: 0.0
2026-01-20 10:41:33 [INFO] orchestrator: Initializing Notion REST client...
2026-01-20 10:41:33 [INFO] orchestrator: ✅ Notion authenticated successfully (REST)
2026-01-20 10:41:33 [INFO] orchestrator:    Bot name: Jupyter
2026-01-20 10:41:33 [INFO] orchestrator:    Bot type: bot
2026-01-20 10:41:33 [INFO] orchestrator: Validating Notion database access (REST)...
2026-01-20 10:41:34 [INFO] orchestrator:    ✅ Literature Database: 'Literature Database' (13 properties)
2026-01-20 10:41:34 [INFO] orchestrator:    ✅ Research Questions Database: 'Research Question' (8 properties)
2026-01-20 10:41:34 [INFO] 

In [15]:
# ============================================================
# Cell 05 — Define orchestration config dataclass and run directory setup
# ============================================================
# Overview:
#   Define OrchestrationConfig dataclass to hold pipeline execution parameters,
#   run directory paths, and step configuration. Implement run directory creation
#   with timestamped folder structure (runs/YYYY-MM-DD_HHMM/) and artifact subdirs.
#   Establish checkpoint and output file path helpers.
#
# Inputs / Outputs:
#   Inputs: env_config (from cell 02), datetime timestamp
#   Outputs: OrchestrationConfig instance, created run directory structure
#
# Notes:
#   - Avoids name collisions by importing dataclasses as "dc"
#   - Run directory: runs/YYYY-MM-DD_HHMM/
#   - Creates subdirectories: artifacts/, reports/, logs/
#   - Disables writes/notifications in dry-run mode
#

from pathlib import Path
from datetime import datetime
from typing import Optional, Dict, Any
import json
import logging
import dataclasses as dc  # <- key fix (avoid "field" name collision)

# Optional: quick guard to detect collisions (safe)
if "field" in globals() and not callable(globals().get("field")):
    logger.warning("⚠️  Detected global name 'field' shadowing dataclasses.field. Using dataclasses alias 'dc' to avoid collisions.")

@dc.dataclass
class OrchestrationConfig:
    """Configuration container for end-to-end pipeline execution."""

    # Run identification
    run_id: str
    run_timestamp: datetime
    dry_run: bool = False

    # Directory paths
    run_dir: Path = dc.field(default=None)
    artifacts_dir: Path = dc.field(default=None)
    reports_dir: Path = dc.field(default=None)
    logs_dir: Path = dc.field(default=None)

    # Step execution flags
    enable_corpus_discovery: bool = True
    enable_pdf_extraction: bool = True
    enable_notion_upload: bool = True
    enable_network_analysis: bool = True
    enable_gap_analysis: bool = True
    enable_daily_scan: bool = True
    enable_rq_update: bool = True

    # Pipeline execution parameters
    max_papers_per_run: int = 50
    timeout_seconds: int = 300
    max_retries: int = 3

    # Report generation flags
    generate_weekly_corpus_report: bool = True
    generate_network_summary: bool = True
    generate_gap_update: bool = True
    generate_rq_proposal: bool = True
    generate_x_draft: bool = True

    # Notion write behavior
    notion_write_enabled: bool = True  # Set False in dry-run
    notion_batch_size: int = 10

    # Google Drive integration
    drive_upload_enabled: bool = False
    drive_folder_id: Optional[str] = None

    # Notification settings
    send_notifications: bool = False
    notification_email: Optional[str] = None
    slack_webhook_url: Optional[str] = None

    # Checkpoint and caching
    use_checkpoints: bool = True
    checkpoint_interval: int = 10

    def __post_init__(self):
        # Create timestamped run directory if not provided
        if self.run_dir is None:
            base_runs_dir = Path("runs")
            timestamp_str = self.run_timestamp.strftime("%Y-%m-%d_%H%M")
            self.run_dir = base_runs_dir / timestamp_str

        self.artifacts_dir = self.run_dir / "artifacts"
        self.reports_dir = self.run_dir / "reports"
        self.logs_dir = self.run_dir / "logs"

        # Disable side effects in dry-run mode
        if self.dry_run:
            self.notion_write_enabled = False
            self.drive_upload_enabled = False
            self.send_notifications = False

    def create_directories(self) -> None:
        for directory in [self.run_dir, self.artifacts_dir, self.reports_dir, self.logs_dir]:
            directory.mkdir(parents=True, exist_ok=True)
            logger.debug(f"Created directory: {directory}")

    def get_artifact_path(self, filename: str) -> Path:
        return self.artifacts_dir / filename

    def get_report_path(self, filename: str) -> Path:
        return self.reports_dir / filename

    def get_log_path(self, filename: str) -> Path:
        return self.logs_dir / filename

    def get_checkpoint_path(self, step_name: str) -> Path:
        return self.artifacts_dir / f"{step_name}_checkpoint.json"

    def to_dict(self) -> Dict[str, Any]:
        return {
            "run_id": self.run_id,
            "run_timestamp": self.run_timestamp.isoformat(),
            "dry_run": self.dry_run,
            "run_dir": str(self.run_dir),
            "step_flags": {
                "corpus_discovery": self.enable_corpus_discovery,
                "pdf_extraction": self.enable_pdf_extraction,
                "notion_upload": self.enable_notion_upload,
                "network_analysis": self.enable_network_analysis,
                "gap_analysis": self.enable_gap_analysis,
                "daily_scan": self.enable_daily_scan,
                "rq_update": self.enable_rq_update,
            },
            "pipeline_params": {
                "max_papers_per_run": self.max_papers_per_run,
                "timeout_seconds": self.timeout_seconds,
                "max_retries": self.max_retries,
            },
            "report_flags": {
                "weekly_corpus_report": self.generate_weekly_corpus_report,
                "network_summary": self.generate_network_summary,
                "gap_update": self.generate_gap_update,
                "rq_proposal": self.generate_rq_proposal,
                "x_draft": self.generate_x_draft,
            },
            "notion_config": {
                "write_enabled": self.notion_write_enabled,
                "batch_size": self.notion_batch_size,
            },
            "drive_config": {
                "upload_enabled": self.drive_upload_enabled,
                "folder_id": self.drive_folder_id,
            },
            "notification_config": {
                "enabled": self.send_notifications,
                "has_email": bool(self.notification_email),
                "has_slack": bool(self.slack_webhook_url),
            },
            "checkpoint_config": {
                "use_checkpoints": self.use_checkpoints,
                "checkpoint_interval": self.checkpoint_interval,
            },
        }


# --- Initialize orchestration configuration ---
logger.info("Initializing orchestration configuration...")

run_timestamp = datetime.now()
run_id = run_timestamp.strftime("%Y%m%d_%H%M%S")

orchestration_config = OrchestrationConfig(
    run_id=run_id,
    run_timestamp=run_timestamp,
    dry_run=DRY_RUN,

    # Pipeline parameters from env_config (note: Cell 02 may expose both names)
    max_papers_per_run=int(getattr(env_config, "arxiv_search_max_results", getattr(env_config, "arxiv_max_results", 50))),
    timeout_seconds=int(getattr(env_config, "timeout_seconds", 300)),
    max_retries=3,

    # Google Drive integration
    drive_upload_enabled=bool(getattr(env_config, "google_drive_folder_id", None)) and not DRY_RUN,
    drive_folder_id=getattr(env_config, "google_drive_folder_id", None),

    # Notification settings
    send_notifications=bool(getattr(env_config, "notification_email", None) or getattr(env_config, "slack_webhook_url", None)) and not DRY_RUN,
    notification_email=getattr(env_config, "notification_email", None),
    slack_webhook_url=getattr(env_config, "slack_webhook_url", None),
)

# Create run directory structure
logger.info(f"Creating run directory: {orchestration_config.run_dir}")
orchestration_config.create_directories()

# Expose run_dir globally (used by Cell 04 manifest save, and later steps)
run_dir = orchestration_config.run_dir

# Configure file logging to run-specific log file
run_log_file = orchestration_config.get_log_path("orchestrator.log")
file_handler = logging.FileHandler(run_log_file)
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(name)s: %(message)s"))
logger.addHandler(file_handler)

logger.info("✅ Orchestration configuration initialized")
logger.info(f"   Run ID: {orchestration_config.run_id}")
logger.info(f"   Run directory: {orchestration_config.run_dir}")
logger.info(f"   Dry-run mode: {orchestration_config.dry_run}")
logger.info(f"   Notion writes: {'DISABLED' if not orchestration_config.notion_write_enabled else 'enabled'}")
logger.info(f"   Drive uploads: {'DISABLED' if not orchestration_config.drive_upload_enabled else 'enabled'}")
logger.info(f"   Notifications: {'DISABLED' if not orchestration_config.send_notifications else 'enabled'}")
logger.info(f"   Logging to: {run_log_file}")

if VERBOSE:
    logger.debug("Full orchestration configuration:")
    logger.debug(json.dumps(orchestration_config.to_dict(), indent=2, default=str))

# --- Define output report file paths ---
REPORT_FILES = {
    "weekly_corpus_update": orchestration_config.get_report_path("weekly_corpus_update_report.md"),
    "network_update_summary": orchestration_config.get_report_path("network_update_summary.md"),
    "gap_update": orchestration_config.get_report_path("gap_update.md"),
    "rq_update_proposal": orchestration_config.get_report_path("rq_update_proposal.md"),
    "x_draft": orchestration_config.get_report_path("x_draft.md"),
}

logger.info("Report output paths configured:")
for report_name, path in REPORT_FILES.items():
    logger.info(f"   {report_name}: {path.name}")

# --- Define checkpoint file paths for each step ---
CHECKPOINT_FILES = {
    "corpus_discovery": orchestration_config.get_checkpoint_path("step_018_corpus_discovery"),
    "pdf_extraction": orchestration_config.get_checkpoint_path("step_019_pdf_extraction"),
    "notion_upload": orchestration_config.get_checkpoint_path("step_020_notion_upload"),
    "network_analysis": orchestration_config.get_checkpoint_path("step_021_network_analysis"),
    "gap_analysis": orchestration_config.get_checkpoint_path("step_022_gap_analysis"),
    "daily_scan": orchestration_config.get_checkpoint_path("step_023_daily_scan"),
    "rq_update": orchestration_config.get_checkpoint_path("step_024_rq_update"),
}

if orchestration_config.use_checkpoints and VERBOSE:
    logger.debug("Checkpoint files:")
    for step_name, path in CHECKPOINT_FILES.items():
        logger.debug(f"   {step_name}: {path.name}")

logger.info("Run directory setup complete. Ready for pipeline execution.")


2026-01-20 10:44:14 [WARNING] orchestrator: ⚠️  Detected global name 'field' shadowing dataclasses.field. Using dataclasses alias 'dc' to avoid collisions.
2026-01-20 10:44:14 [INFO] orchestrator: Initializing orchestration configuration...
2026-01-20 10:44:14 [INFO] orchestrator: Creating run directory: runs/2026-01-20_1044
2026-01-20 10:44:14 [INFO] orchestrator: ✅ Orchestration configuration initialized
2026-01-20 10:44:14 [INFO] orchestrator:    Run ID: 20260120_104414
2026-01-20 10:44:14 [INFO] orchestrator:    Run directory: runs/2026-01-20_1044
2026-01-20 10:44:14 [INFO] orchestrator:    Dry-run mode: False
2026-01-20 10:44:14 [INFO] orchestrator:    Notion writes: enabled
2026-01-20 10:44:14 [INFO] orchestrator:    Drive uploads: DISABLED
2026-01-20 10:44:14 [INFO] orchestrator:    Notifications: DISABLED
2026-01-20 10:44:14 [INFO] orchestrator:    Logging to: runs/2026-01-20_1044/logs/orchestrator.log
2026-01-20 10:44:14 [INFO] orchestrator: Report output paths configured:
202

In [17]:
# ============================================================
# Cell 06 — Implement step function placeholders for notebooks 018-024
# ============================================================
# Overview:
#   Define placeholder step functions for each of the seven prior notebooks
#   (018-024). Each function returns a StepResult with success/failure status,
#   artifacts dictionary, and error information. Uses checkpoint files for
#   recovery and progress tracking.
#
# Inputs / Outputs:
#   Inputs: orchestration_config, clients (openai_client, notion_rest), CHECKPOINT_FILES
#   Outputs: StepResult instances with execution metadata and artifacts
#
# Notes:
#   - Avoids dataclasses "field" collisions by importing dataclasses as dc
#   - All functions are placeholders with TODO markers
#   - Checkpoint I/O is real (writes JSON), but step bodies do no network calls yet
#

from typing import Dict, Any, Optional, List
from pathlib import Path
from datetime import datetime
import json
import dataclasses as dc  # <- critical fix

if "field" in globals() and not callable(globals().get("field")):
    logger.warning("⚠️  Detected global name 'field' shadowing dataclasses.field. Using dataclasses alias 'dc' to avoid collisions.")

# --- Step result container ---
@dc.dataclass
class StepResult:
    """Container for step execution results."""
    step_name: str
    success: bool
    duration_seconds: float
    artifacts: Dict[str, Any] = dc.field(default_factory=dict)
    error: Optional[str] = None
    checkpoint_path: Optional[Path] = None


# --- Helper: Save/load checkpoint ---
def save_checkpoint(checkpoint_path: Path, data: Dict[str, Any]) -> None:
    """Save checkpoint data to JSON file."""
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    checkpoint_path.write_text(json.dumps(data, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    logger.debug(f"Checkpoint saved: {checkpoint_path}")


def load_checkpoint(checkpoint_path: Path) -> Optional[Dict[str, Any]]:
    """Load checkpoint data from JSON file if exists."""
    if checkpoint_path.exists():
        return json.loads(checkpoint_path.read_text(encoding="utf-8"))
    return None


# --- Step 018: Corpus discovery (arXiv + Semantic Scholar) ---
def step_018_corpus_discovery(config: OrchestrationConfig) -> StepResult:
    """Execute notebook 018: seed corpus discovery via arXiv/S2 API."""
    start_time = datetime.now()
    checkpoint_path = CHECKPOINT_FILES["corpus_discovery"]

    # TODO:
    # - Load checkpoint if exists
    # - Execute arXiv search with query from env
    # - Execute Semantic Scholar search
    # - Deduplicate and filter results
    # - Save checkpoint with discovered papers

    artifacts = {"papers_discovered": 0, "sources": ["arXiv", "SemanticScholar"]}
    duration = (datetime.now() - start_time).total_seconds()
    return StepResult("corpus_discovery", True, duration, artifacts, checkpoint_path=checkpoint_path)


# --- Step 019: PDF extraction ---
def step_019_pdf_extraction(config: OrchestrationConfig, papers: List[Dict[str, Any]]) -> StepResult:
    """Execute notebook 019: download PDFs and extract text/metadata."""
    start_time = datetime.now()
    checkpoint_path = CHECKPOINT_FILES["pdf_extraction"]

    # TODO:
    # - Download PDFs for discovered papers
    # - Extract text via PyPDF2/pdfplumber
    # - Extract metadata (title, authors, abstract)
    # - Save checkpoint with extraction results

    artifacts = {"pdfs_downloaded": 0, "extractions_complete": 0}
    duration = (datetime.now() - start_time).total_seconds()
    return StepResult("pdf_extraction", True, duration, artifacts, checkpoint_path=checkpoint_path)


# --- Step 020: Notion upload ---
def step_020_notion_upload(config: OrchestrationConfig, papers: List[Dict[str, Any]]) -> StepResult:
    """Execute notebook 020: upload papers to Notion LIT_DB."""
    start_time = datetime.now()
    checkpoint_path = CHECKPOINT_FILES["notion_upload"]

    # TODO:
    # - Batch papers into notion_batch_size chunks
    # - Create Notion pages with LIT_SCHEMA mapping (requests-based)
    # - Handle duplicates (check existing entries)
    # - Save checkpoint after each batch

    artifacts = {"pages_created": 0, "duplicates_skipped": 0}
    duration = (datetime.now() - start_time).total_seconds()
    return StepResult("notion_upload", True, duration, artifacts, checkpoint_path=checkpoint_path)


# --- Step 021: Network analysis ---
def step_021_network_analysis(config: OrchestrationConfig) -> StepResult:
    """Execute notebook 021: citation network analysis."""
    start_time = datetime.now()
    checkpoint_path = CHECKPOINT_FILES["network_analysis"]

    # TODO:
    # - Fetch citation graph (OpenAlex/S2)
    # - Build networkx graph
    # - Compute centrality metrics
    # - Identify key papers and clusters

    artifacts = {"nodes": 0, "edges": 0, "clusters": 0}
    duration = (datetime.now() - start_time).total_seconds()
    return StepResult("network_analysis", True, duration, artifacts, checkpoint_path=checkpoint_path)


# --- Step 022: Gap analysis ---
def step_022_gap_analysis(config: OrchestrationConfig) -> StepResult:
    """Execute notebook 022: research gap detection."""
    start_time = datetime.now()
    checkpoint_path = CHECKPOINT_FILES["gap_analysis"]

    # TODO:
    # - Query Notion RQ_DB for active questions
    # - Map literature to RQ coverage
    # - Identify under-researched themes
    # - Generate gap summary with LLM

    artifacts = {"gaps_identified": 0, "rqs_analyzed": 0}
    duration = (datetime.now() - start_time).total_seconds()
    return StepResult("gap_analysis", True, duration, artifacts, checkpoint_path=checkpoint_path)


# --- Step 023: Daily scan ---
def step_023_daily_scan(config: OrchestrationConfig) -> StepResult:
    """Execute notebook 023: daily new papers scan."""
    start_time = datetime.now()
    checkpoint_path = CHECKPOINT_FILES["daily_scan"]

    # TODO:
    # - Search arXiv for papers from last 24h
    # - Filter by relevance threshold
    # - Quick-add to Notion with minimal metadata

    artifacts = {"new_papers": 0, "relevant_papers": 0}
    duration = (datetime.now() - start_time).total_seconds()
    return StepResult("daily_scan", True, duration, artifacts, checkpoint_path=checkpoint_path)


# --- Step 024: RQ update ---
def step_024_rq_update(config: OrchestrationConfig) -> StepResult:
    """Execute notebook 024: RQ refresh and answer updates."""
    start_time = datetime.now()
    checkpoint_path = CHECKPOINT_FILES["rq_update"]

    # TODO:
    # - Query RQ_DB for stale questions
    # - Fetch related literature added since last review
    # - Generate answer updates with LLM
    # - Update RQ confidence levels

    artifacts = {"rqs_updated": 0, "new_evidence": 0}
    duration = (datetime.now() - start_time).total_seconds()
    return StepResult("rq_update", True, duration, artifacts, checkpoint_path=checkpoint_path)


logger.info("✅ Step function placeholders defined for notebooks 018-024")
logger.info("   All steps return StepResult with artifacts and checkpoint support")


2026-01-20 10:47:30 [WARNING] orchestrator: ⚠️  Detected global name 'field' shadowing dataclasses.field. Using dataclasses alias 'dc' to avoid collisions.
2026-01-20 10:47:30 [INFO] orchestrator: ✅ Step function placeholders defined for notebooks 018-024
2026-01-20 10:47:30 [INFO] orchestrator:    All steps return StepResult with artifacts and checkpoint support


In [18]:
# ============================================================
# Cell 07 — Implement failure handling and notification stubs
# ============================================================
# Overview:
#   Define failure handling logic for pipeline steps and notification
#   functions for alerting on errors. Implements step retry logic,
#   error aggregation, and notification delivery stubs (email, Slack).
#
# Inputs / Outputs:
#   Inputs: StepResult objects with error information
#   Outputs: Logged errors, notification delivery status
#
# Notes:
#   - Notification functions are stubs with TODO markers
#   - Supports retry logic with exponential backoff
#   - Aggregates errors for batch notification
#

# --- Error aggregation container ---
@dataclass
class PipelineFailure:
    """Container for pipeline failure information."""
    step_name: str
    error_message: str
    timestamp: datetime
    stack_trace: Optional[str] = None
    retry_count: int = 0


# --- Retry logic with exponential backoff ---
def retry_step(step_func, config: OrchestrationConfig, *args, **kwargs) -> StepResult:
    """Execute step with retry logic."""
    max_retries = config.max_retries
    
    for attempt in range(max_retries + 1):
        try:
            result = step_func(config, *args, **kwargs)
            if result.success:
                return result
            logger.warning(f"Step {result.step_name} failed (attempt {attempt + 1}/{max_retries + 1})")
        except Exception as e:
            logger.error(f"Exception in {step_func.__name__} (attempt {attempt + 1}): {e}")
            if attempt == max_retries:
                # Final failure
                return StepResult(
                    step_name=step_func.__name__,
                    success=False,
                    duration_seconds=0.0,
                    error=str(e)
                )
            # TODO: Add exponential backoff delay here
    
    # Should not reach here
    return StepResult(step_name=step_func.__name__, success=False, duration_seconds=0.0, error="Max retries exceeded")


# --- Notification delivery stubs ---
def send_email_notification(subject: str, body: str, recipient: str) -> bool:
    """Send email notification (stub)."""
    # TODO: Implement SMTP email sending
    logger.info(f"[STUB] Would send email to {recipient}: {subject}")
    return True


def send_slack_notification(message: str, webhook_url: str) -> bool:
    """Send Slack notification (stub)."""
    # TODO: Implement Slack webhook POST
    logger.info(f"[STUB] Would send Slack message to {webhook_url[:30]}...")
    return True


# --- Failure notification dispatcher ---
def notify_failure(failures: List[PipelineFailure], config: OrchestrationConfig) -> None:
    """Send failure notifications via configured channels."""
    if not config.send_notifications or not failures:
        return
    
    # Build notification message
    message_lines = [
        f"Pipeline run {config.run_id} encountered {len(failures)} failure(s):",
        ""
    ]
    for failure in failures:
        message_lines.append(f"- {failure.step_name}: {failure.error_message}")
    
    message = "\n".join(message_lines)
    
    # Send via email
    if config.notification_email:
        send_email_notification(
            subject=f"Pipeline Failure: {config.run_id}",
            body=message,
            recipient=config.notification_email
        )
    
    # Send via Slack
    if config.slack_webhook_url:
        send_slack_notification(message, config.slack_webhook_url)


logger.info("✅ Failure handling and notification stubs defined")


2026-01-20 10:47:39 [INFO] orchestrator: ✅ Failure handling and notification stubs defined


In [19]:
# ============================================================
# Cell 08 — Define report compilation functions for five output documents
# ============================================================
# Overview:
#   Implement report compilation functions that aggregate results from
#   pipeline steps and generate five markdown output documents:
#   1. Weekly corpus update report
#   2. Network update summary
#   3. Gap analysis update
#   4. RQ update proposal
#   5. X (Twitter) draft post
#
# Inputs / Outputs:
#   Inputs: StepResult artifacts, orchestration_config
#   Outputs: Markdown report files written to reports/ directory
#
# Notes:
#   - Each function builds a markdown document from step artifacts
#   - Reports are written to REPORT_FILES paths defined in cell 05
#   - Includes timestamp and run metadata in each report header
#   - Uses simple string templates; no complex formatting
#

# --- Report header generator ---
def generate_report_header(title: str, config: OrchestrationConfig) -> str:
    """Generate standard markdown header for reports."""
    return f"""# {title}

**Run ID:** {config.run_id}  
**Timestamp:** {config.run_timestamp.strftime('%Y-%m-%d %H:%M:%S')}  
**Mode:** {'DRY RUN' if config.dry_run else 'Production'}

---

"""


# --- 1. Weekly corpus update report ---
def compile_weekly_corpus_report(step_results: Dict[str, StepResult], config: OrchestrationConfig) -> str:
    """Compile weekly corpus update report from steps 018-020."""
    report = generate_report_header("Weekly Corpus Update Report", config)
    
    # TODO: Extract metrics from corpus_discovery, pdf_extraction, notion_upload steps
    # TODO: Add sections: Papers Discovered, PDFs Processed, Notion Pages Created
    # TODO: Include source breakdown (arXiv vs SemanticScholar)
    # TODO: List any failures or skipped papers
    
    report += "## Summary\n\n"
    report += "- Papers discovered: [TODO]\n"
    report += "- PDFs extracted: [TODO]\n"
    report += "- Notion pages created: [TODO]\n\n"
    
    return report


# --- 2. Network update summary ---
def compile_network_summary(step_results: Dict[str, StepResult], config: OrchestrationConfig) -> str:
    """Compile citation network analysis summary from step 021."""
    report = generate_report_header("Citation Network Update Summary", config)
    
    # TODO: Extract network metrics (nodes, edges, clusters)
    # TODO: Identify top central papers
    # TODO: Highlight new connections since last run
    
    report += "## Network Metrics\n\n"
    report += "- Total nodes: [TODO]\n"
    report += "- Total edges: [TODO]\n\n"
    
    return report


# --- 3. Gap analysis update ---
def compile_gap_update(step_results: Dict[str, StepResult], config: OrchestrationConfig) -> str:
    """Compile research gap analysis from step 022."""
    report = generate_report_header("Research Gap Analysis Update", config)
    
    # TODO: List identified gaps from step artifacts
    # TODO: Map gaps to under-covered RQs
    # TODO: Suggest new research directions
    
    report += "## Identified Gaps\n\n"
    report += "[TODO: List gaps with evidence counts]\n\n"
    
    return report


# --- 4. RQ update proposal ---
def compile_rq_proposal(step_results: Dict[str, StepResult], config: OrchestrationConfig) -> str:
    """Compile RQ update proposal from step 024."""
    report = generate_report_header("Research Questions Update Proposal", config)
    
    # TODO: List RQs with proposed answer updates
    # TODO: Include new evidence citations
    # TODO: Show confidence level changes
    
    report += "## Updated RQs\n\n"
    report += "[TODO: List RQ updates with evidence]\n\n"
    
    return report


# --- 5. X (Twitter) draft post ---
def compile_x_draft(step_results: Dict[str, StepResult], config: OrchestrationConfig) -> str:
    """Compile X/Twitter thread draft summarizing weekly updates."""
    report = generate_report_header("X Thread Draft", config)
    
    # TODO: Generate 280-char summary tweets
    # TODO: Highlight top 3 papers discovered
    # TODO: Include key finding from gap analysis
    
    report += "## Thread (max 280 chars per tweet)\n\n"
    report += "1/ [TODO: Opening tweet with weekly summary]\n\n"
    report += "2/ [TODO: Top papers highlight]\n\n"
    
    return report


# --- Report writer ---
def write_all_reports(step_results: Dict[str, StepResult], config: OrchestrationConfig) -> Dict[str, bool]:
    """Generate and write all five reports to disk."""
    logger.info("Compiling final reports...")
    
    report_status = {}
    
    if config.generate_weekly_corpus_report:
        content = compile_weekly_corpus_report(step_results, config)
        REPORT_FILES['weekly_corpus_update'].write_text(content)
        report_status['weekly_corpus_update'] = True
        logger.info(f"  ✅ {REPORT_FILES['weekly_corpus_update'].name}")
    
    if config.generate_network_summary:
        content = compile_network_summary(step_results, config)
        REPORT_FILES['network_update_summary'].write_text(content)
        report_status['network_update_summary'] = True
        logger.info(f"  ✅ {REPORT_FILES['network_update_summary'].name}")
    
    if config.generate_gap_update:
        content = compile_gap_update(step_results, config)
        REPORT_FILES['gap_update'].write_text(content)
        report_status['gap_update'] = True
        logger.info(f"  ✅ {REPORT_FILES['gap_update'].name}")
    
    if config.generate_rq_proposal:
        content = compile_rq_proposal(step_results, config)
        REPORT_FILES['rq_update_proposal'].write_text(content)
        report_status['rq_update_proposal'] = True
        logger.info(f"  ✅ {REPORT_FILES['rq_update_proposal'].name}")
    
    if config.generate_x_draft:
        content = compile_x_draft(step_results, config)
        REPORT_FILES['x_draft'].write_text(content)
        report_status['x_draft'] = True
        logger.info(f"  ✅ {REPORT_FILES['x_draft'].name}")
    
    return report_status


logger.info("✅ Report compilation functions defined for five output documents")


2026-01-20 10:47:48 [INFO] orchestrator: ✅ Report compilation functions defined for five output documents


In [20]:
# ============================================================
# Cell 09 — Implement main pipeline orchestrator function
# ============================================================
# Overview:
#   Main orchestrator function that executes the full pipeline sequence:
#   corpus discovery → PDF extraction → Notion upload → network analysis →
#   gap analysis → daily scan → RQ update. Tracks step results, handles
#   failures, and returns aggregated execution summary.
#
# Inputs / Outputs:
#   Inputs: OrchestrationConfig instance
#   Outputs: Dict with step results, failures, and execution metadata
#
# Notes:
#   - Executes steps conditionally based on config enable flags
#   - Collects StepResult objects for downstream report compilation
#   - Aggregates failures for notification
#   - Uses retry logic for resilience
#

# --- Main orchestrator function ---
def run_pipeline(config: OrchestrationConfig) -> Dict[str, Any]:
    """
    Execute full end-to-end pipeline with all seven steps.
    Returns execution summary with results and failures.
    """
    logger.info("="*60)
    logger.info(f"Starting pipeline run: {config.run_id}")
    logger.info(f"Mode: {'DRY RUN' if config.dry_run else 'PRODUCTION'}")
    logger.info("="*60)
    
    step_results: Dict[str, StepResult] = {}
    failures: List[PipelineFailure] = []
    start_time = datetime.now()
    
    # --- Weekly corpus update workflow (steps 018-020) ---
    discovered_papers = []
    
    if config.enable_corpus_discovery:
        logger.info("\n[Step 1/7] Corpus discovery (arXiv + Semantic Scholar)...")
        result = retry_step(step_018_corpus_discovery, config)
        step_results['corpus_discovery'] = result
        if not result.success:
            failures.append(PipelineFailure('corpus_discovery', result.error or 'Unknown error', datetime.now()))
        else:
            # TODO: Extract discovered_papers from result.artifacts
            pass
    
    if config.enable_pdf_extraction and discovered_papers:
        logger.info("\n[Step 2/7] PDF extraction and text parsing...")
        result = retry_step(step_019_pdf_extraction, config, discovered_papers)
        step_results['pdf_extraction'] = result
        if not result.success:
            failures.append(PipelineFailure('pdf_extraction', result.error or 'Unknown error', datetime.now()))
    
    if config.enable_notion_upload and discovered_papers:
        logger.info("\n[Step 3/7] Notion database upload...")
        result = retry_step(step_020_notion_upload, config, discovered_papers)
        step_results['notion_upload'] = result
        if not result.success:
            failures.append(PipelineFailure('notion_upload', result.error or 'Unknown error', datetime.now()))
    
    # --- Analysis workflow (steps 021-022) ---
    if config.enable_network_analysis:
        logger.info("\n[Step 4/7] Citation network analysis...")
        result = retry_step(step_021_network_analysis, config)
        step_results['network_analysis'] = result
        if not result.success:
            failures.append(PipelineFailure('network_analysis', result.error or 'Unknown error', datetime.now()))
    
    if config.enable_gap_analysis:
        logger.info("\n[Step 5/7] Research gap detection...")
        result = retry_step(step_022_gap_analysis, config)
        step_results['gap_analysis'] = result
        if not result.success:
            failures.append(PipelineFailure('gap_analysis', result.error or 'Unknown error', datetime.now()))
    
    # --- Daily maintenance workflow (steps 023-024) ---
    if config.enable_daily_scan:
        logger.info("\n[Step 6/7] Daily new papers scan...")
        result = retry_step(step_023_daily_scan, config)
        step_results['daily_scan'] = result
        if not result.success:
            failures.append(PipelineFailure('daily_scan', result.error or 'Unknown error', datetime.now()))
    
    if config.enable_rq_update:
        logger.info("\n[Step 7/7] Research questions update...")
        result = retry_step(step_024_rq_update, config)
        step_results['rq_update'] = result
        if not result.success:
            failures.append(PipelineFailure('rq_update', result.error or 'Unknown error', datetime.now()))
    
    # --- Pipeline completion ---
    duration = (datetime.now() - start_time).total_seconds()
    
    logger.info("\n" + "="*60)
    logger.info(f"Pipeline completed in {duration:.1f}s")
    logger.info(f"Steps executed: {len(step_results)}/{7}")
    logger.info(f"Failures: {len(failures)}")
    logger.info("="*60)
    
    # Send failure notifications if any
    if failures:
        notify_failure(failures, config)
    
    return {
        'run_id': config.run_id,
        'success': len(failures) == 0,
        'duration_seconds': duration,
        'step_results': step_results,
        'failures': [asdict(f) for f in failures],
        'steps_executed': len(step_results),
        'timestamp': datetime.now().isoformat(),
    }


logger.info("✅ Main pipeline orchestrator function defined")
logger.info("   Executes all seven steps with retry logic and failure tracking")


2026-01-20 10:47:56 [INFO] orchestrator: ✅ Main pipeline orchestrator function defined
2026-01-20 10:47:56 [INFO] orchestrator:    Executes all seven steps with retry logic and failure tracking


In [21]:
# ============================================================
# Cell 10 — Execute weekly corpus update steps (018-020)
# ============================================================
# Overview:
#   Execute the first three pipeline steps that constitute the weekly
#   corpus update workflow: (1) discover new papers via arXiv/Semantic Scholar,
#   (2) extract PDFs and parse text, (3) upload to Notion Literature DB.
#   Demonstrates end-to-end integration of steps with data flow, checkpoint
#   handling, and result aggregation.
#
# Inputs / Outputs:
#   Inputs: orchestration_config (from cell 05), global clients
#   Outputs: Executed step results, discovered_papers list, checkpoint files
#
# Notes:
#   - Uses retry_step wrapper for resilience
#   - Passes discovered_papers between steps for data flow
#   - Logs progress and timing for each step
#   - Creates checkpoint files in artifacts/ directory
#   - Can be run independently for testing weekly corpus updates
#

# --- Initialize tracking for weekly corpus update ---
logger.info("\n" + "="*60)
logger.info("WEEKLY CORPUS UPDATE WORKFLOW (Steps 018-020)")
logger.info("="*60)

weekly_step_results: Dict[str, StepResult] = {}
weekly_failures: List[PipelineFailure] = []
weekly_start_time = datetime.now()

# --- Step 1/3: Corpus Discovery (Step 018) ---
logger.info("\n[Weekly Step 1/3] Corpus discovery via arXiv + Semantic Scholar...")
logger.info(f"  Max papers per source: {orchestration_config.max_papers_per_run}")

discovered_papers: List[Dict[str, Any]] = []

if orchestration_config.enable_corpus_discovery:
    try:
        # Execute corpus discovery step with retry logic
        corpus_result = retry_step(step_018_corpus_discovery, orchestration_config)
        weekly_step_results['corpus_discovery'] = corpus_result
        
        if corpus_result.success:
            # Extract discovered papers from artifacts
            # TODO: In actual implementation, this would contain:
            # - arXiv search results with metadata (title, authors, abstract, arxiv_id)
            # - Semantic Scholar results with S2 IDs and citation counts
            # - Deduplicated and merged paper list
            
            papers_count = corpus_result.artifacts.get('papers_discovered', 0)
            sources = corpus_result.artifacts.get('sources', [])
            
            logger.info(f"  ✅ Corpus discovery completed")
            logger.info(f"     Papers discovered: {papers_count}")
            logger.info(f"     Sources: {', '.join(sources)}")
            logger.info(f"     Duration: {corpus_result.duration_seconds:.1f}s")
            
            # TODO: Load actual papers from checkpoint file
            # For now, create placeholder structure
            # discovered_papers = load_checkpoint(corpus_result.checkpoint_path).get('papers', [])
            
            # Placeholder: simulate discovered papers structure
            discovered_papers = [
                {
                    'title': f'[Placeholder Paper {i+1}]',
                    'authors': [],
                    'abstract': '',
                    'year': 2024,
                    'source': 'arXiv' if i % 2 == 0 else 'SemanticScholar',
                    'url': '',
                    'pdf_url': '',
                    'arxiv_id': f'2024.{i:05d}' if i % 2 == 0 else None,
                    's2_id': f's2_{i}' if i % 2 == 1 else None,
                }
                for i in range(papers_count)
            ]
            
        else:
            # Step failed
            error_msg = corpus_result.error or 'Unknown error in corpus discovery'
            logger.error(f"  ❌ Corpus discovery failed: {error_msg}")
            weekly_failures.append(
                PipelineFailure(
                    step_name='corpus_discovery',
                    error_message=error_msg,
                    timestamp=datetime.now()
                )
            )
            
    except Exception as e:
        error_msg = f"Exception during corpus discovery: {str(e)}"
        logger.error(f"  ❌ {error_msg}")
        logger.debug(traceback.format_exc())
        
        weekly_failures.append(
            PipelineFailure(
                step_name='corpus_discovery',
                error_message=error_msg,
                timestamp=datetime.now(),
                stack_trace=traceback.format_exc()
            )
        )
else:
    logger.info("  ⏭️  Corpus discovery disabled in config")


# --- Step 2/3: PDF Extraction (Step 019) ---
logger.info("\n[Weekly Step 2/3] PDF extraction and text parsing...")

if orchestration_config.enable_pdf_extraction and discovered_papers:
    logger.info(f"  Processing {len(discovered_papers)} papers...")
    
    try:
        # Execute PDF extraction step
        pdf_result = retry_step(
            step_019_pdf_extraction,
            orchestration_config,
            discovered_papers
        )
        weekly_step_results['pdf_extraction'] = pdf_result
        
        if pdf_result.success:
            pdfs_downloaded = pdf_result.artifacts.get('pdfs_downloaded', 0)
            extractions_complete = pdf_result.artifacts.get('extractions_complete', 0)
            
            logger.info(f"  ✅ PDF extraction completed")
            logger.info(f"     PDFs downloaded: {pdfs_downloaded}")
            logger.info(f"     Extractions complete: {extractions_complete}")
            logger.info(f"     Duration: {pdf_result.duration_seconds:.1f}s")
            
            # TODO: Enrich discovered_papers with extracted text
            # In actual implementation:
            # - Download PDFs to local cache
            # - Extract text via PyPDF2/pdfplumber
            # - Extract metadata (author list, sections, references)
            # - Upload PDFs to Google Drive if enabled
            # - Update discovered_papers list with extraction results
            
        else:
            error_msg = pdf_result.error or 'Unknown error in PDF extraction'
            logger.error(f"  ❌ PDF extraction failed: {error_msg}")
            weekly_failures.append(
                PipelineFailure(
                    step_name='pdf_extraction',
                    error_message=error_msg,
                    timestamp=datetime.now()
                )
            )
            
    except Exception as e:
        error_msg = f"Exception during PDF extraction: {str(e)}"
        logger.error(f"  ❌ {error_msg}")
        logger.debug(traceback.format_exc())
        
        weekly_failures.append(
            PipelineFailure(
                step_name='pdf_extraction',
                error_message=error_msg,
                timestamp=datetime.now(),
                stack_trace=traceback.format_exc()
            )
        )
        
elif not orchestration_config.enable_pdf_extraction:
    logger.info("  ⏭️  PDF extraction disabled in config")
else:
    logger.info("  ⏭️  No papers to process (corpus discovery failed or returned 0 papers)")


# --- Step 3/3: Notion Upload (Step 020) ---
logger.info("\n[Weekly Step 3/3] Upload to Notion Literature Database...")

if orchestration_config.enable_notion_upload and discovered_papers:
    logger.info(f"  Uploading {len(discovered_papers)} papers to Notion...")
    logger.info(f"  Batch size: {orchestration_config.notion_batch_size}")
    logger.info(f"  Write mode: {'DISABLED (dry-run)' if not orchestration_config.notion_write_enabled else 'ENABLED'}")
    
    try:
        # Execute Notion upload step
        notion_result = retry_step(
            step_020_notion_upload,
            orchestration_config,
            discovered_papers
        )
        weekly_step_results['notion_upload'] = notion_result
        
        if notion_result.success:
            pages_created = notion_result.artifacts.get('pages_created', 0)
            duplicates_skipped = notion_result.artifacts.get('duplicates_skipped', 0)
            
            logger.info(f"  ✅ Notion upload completed")
            logger.info(f"     Pages created: {pages_created}")
            logger.info(f"     Duplicates skipped: {duplicates_skipped}")
            logger.info(f"     Duration: {notion_result.duration_seconds:.1f}s")
            
            # TODO: In actual implementation:
            # - Batch papers into chunks of notion_batch_size
            # - For each paper, check for duplicates (by arXiv ID or title)
            # - Create Notion page with LitDBSchema properties
            # - Link to Google Drive PDF if available
            # - Set initial status, relevance, themes (via LLM classification)
            # - Save checkpoint after each batch for resume capability
            
            if orchestration_config.dry_run:
                logger.warning("     (No actual writes performed in dry-run mode)")
                
        else:
            error_msg = notion_result.error or 'Unknown error in Notion upload'
            logger.error(f"  ❌ Notion upload failed: {error_msg}")
            weekly_failures.append(
                PipelineFailure(
                    step_name='notion_upload',
                    error_message=error_msg,
                    timestamp=datetime.now()
                )
            )
            
    except Exception as e:
        error_msg = f"Exception during Notion upload: {str(e)}"
        logger.error(f"  ❌ {error_msg}")
        logger.debug(traceback.format_exc())
        
        weekly_failures.append(
            PipelineFailure(
                step_name='notion_upload',
                error_message=error_msg,
                timestamp=datetime.now(),
                stack_trace=traceback.format_exc()
            )
        )
        
elif not orchestration_config.enable_notion_upload:
    logger.info("  ⏭️  Notion upload disabled in config")
else:
    logger.info("  ⏭️  No papers to upload (previous steps failed or returned 0 papers)")


# --- Weekly workflow summary ---
weekly_duration = (datetime.now() - weekly_start_time).total_seconds()

logger.info("\n" + "="*60)
logger.info("WEEKLY CORPUS UPDATE WORKFLOW COMPLETED")
logger.info(f"Total duration: {weekly_duration:.1f}s")
logger.info(f"Steps executed: {len(weekly_step_results)}/3")
logger.info(f"Failures: {len(weekly_failures)}")

if discovered_papers:
    logger.info(f"Papers processed: {len(discovered_papers)}")

if weekly_failures:
    logger.warning("\nFailures encountered:")
    for failure in weekly_failures:
        logger.warning(f"  - {failure.step_name}: {failure.error_message}")
else:
    logger.info("\n✅ All weekly corpus update steps completed successfully")

logger.info("="*60)

# Store results for downstream use in cells 13-14
weekly_corpus_summary = {
    'step_results': weekly_step_results,
    'failures': [asdict(f) for f in weekly_failures],
    'duration_seconds': weekly_duration,
    'papers_discovered': len(discovered_papers),
    'success': len(weekly_failures) == 0,
}

if VERBOSE:
    logger.debug("\nWeekly corpus update summary:")
    logger.debug(json.dumps(weekly_corpus_summary, indent=2, default=str))


2026-01-20 10:48:13 [INFO] orchestrator: 
2026-01-20 10:48:13 [INFO] orchestrator: WEEKLY CORPUS UPDATE WORKFLOW (Steps 018-020)
2026-01-20 10:48:13 [INFO] orchestrator: ============================================================
2026-01-20 10:48:13 [INFO] orchestrator: 
[Weekly Step 1/3] Corpus discovery via arXiv + Semantic Scholar...
2026-01-20 10:48:13 [INFO] orchestrator:   Max papers per source: 50
2026-01-20 10:48:13 [INFO] orchestrator:   ✅ Corpus discovery completed
2026-01-20 10:48:13 [INFO] orchestrator:      Papers discovered: 0
2026-01-20 10:48:13 [INFO] orchestrator:      Sources: arXiv, SemanticScholar
2026-01-20 10:48:13 [INFO] orchestrator:      Duration: 0.0s
2026-01-20 10:48:13 [INFO] orchestrator: 
[Weekly Step 2/3] PDF extraction and text parsing...
2026-01-20 10:48:13 [INFO] orchestrator:   ⏭️  No papers to process (corpus discovery failed or returned 0 papers)
2026-01-20 10:48:13 [INFO] orchestrator: 
[Weekly Step 3/3] Upload to Notion Literature Database...
202

In [22]:
# ============================================================
# Cell 11 — Execute network and gap analysis steps (021-022)
# ============================================================
# Overview:
#   Execute the network analysis and gap detection workflow: (4) build citation
#   network from Semantic Scholar data and compute centrality metrics, (5) identify
#   research gaps by mapping literature coverage to active RQs. Demonstrates
#   analysis pipeline integration with Notion database queries and LLM-assisted
#   gap identification.
#
# Inputs / Outputs:
#   Inputs: orchestration_config, notion_client, openai_client, LIT_DB and RQ_DB data
#   Outputs: Executed step results, network metrics, gap analysis artifacts
#
# Notes:
#   - Network analysis requires Semantic Scholar API calls (implement in TODO)
#   - Gap analysis queries Notion RQ_DB to map literature coverage
#   - Uses LLM to synthesize gap narratives from coverage data
#   - Creates checkpoint files for resume capability
#   - Can be run independently after weekly corpus update completes
#

# --- Initialize tracking for analysis workflow ---
logger.info("\n" + "="*60)
logger.info("ANALYSIS WORKFLOW (Steps 021-022)")
logger.info("="*60)

analysis_step_results: Dict[str, StepResult] = {}
analysis_failures: List[PipelineFailure] = []
analysis_start_time = datetime.now()

# --- Step 4/7: Citation Network Analysis (Step 021) ---
logger.info("\n[Analysis Step 1/2] Citation network analysis...")

network_metrics: Dict[str, Any] = {}

if orchestration_config.enable_network_analysis:
    try:
        # Execute network analysis step with retry logic
        network_result = retry_step(step_021_network_analysis, orchestration_config)
        analysis_step_results['network_analysis'] = network_result
        
        if network_result.success:
            # Extract network metrics from artifacts
            nodes_count = network_result.artifacts.get('nodes', 0)
            edges_count = network_result.artifacts.get('edges', 0)
            clusters_count = network_result.artifacts.get('clusters', 0)
            
            logger.info(f"  ✅ Network analysis completed")
            logger.info(f"     Network nodes: {nodes_count}")
            logger.info(f"     Citation edges: {edges_count}")
            logger.info(f"     Identified clusters: {clusters_count}")
            logger.info(f"     Duration: {network_result.duration_seconds:.1f}s")
            
            # TODO: In actual implementation, this step would:
            # - Query Notion LIT_DB for all papers with S2 IDs
            # - Fetch citation graph from Semantic Scholar API
            # - Build networkx graph with papers as nodes, citations as edges
            # - Compute centrality metrics:
            #   * PageRank (identify influential papers)
            #   * Betweenness centrality (find bridging papers)
            #   * Degree centrality (most cited/citing papers)
            # - Detect communities/clusters using Louvain or similar
            # - Identify key papers per cluster
            # - Track network evolution (compare to previous checkpoint)
            # - Save network graph and metrics to checkpoint
            
            network_metrics = network_result.artifacts.copy()
            
            # TODO: Load detailed network data from checkpoint
            # checkpoint_data = load_checkpoint(network_result.checkpoint_path)
            # network_metrics['top_papers'] = checkpoint_data.get('top_papers', [])
            # network_metrics['cluster_themes'] = checkpoint_data.get('cluster_themes', {})
            
            # Placeholder: simulate network structure
            network_metrics.update({
                'top_papers_by_centrality': [
                    {'title': f'[High-impact paper {i+1}]', 'centrality_score': 0.9 - i*0.1}
                    for i in range(min(5, nodes_count))
                ],
                'cluster_themes': {
                    f'Cluster {i+1}': f'[Theme {i+1}]' for i in range(clusters_count)
                },
            })
            
            if VERBOSE:
                logger.debug(f"\nNetwork metrics summary:")
                logger.debug(json.dumps(network_metrics, indent=2, default=str))
                
        else:
            # Step failed
            error_msg = network_result.error or 'Unknown error in network analysis'
            logger.error(f"  ❌ Network analysis failed: {error_msg}")
            analysis_failures.append(
                PipelineFailure(
                    step_name='network_analysis',
                    error_message=error_msg,
                    timestamp=datetime.now()
                )
            )
            
    except Exception as e:
        error_msg = f"Exception during network analysis: {str(e)}"
        logger.error(f"  ❌ {error_msg}")
        logger.debug(traceback.format_exc())
        
        analysis_failures.append(
            PipelineFailure(
                step_name='network_analysis',
                error_message=error_msg,
                timestamp=datetime.now(),
                stack_trace=traceback.format_exc()
            )
        )
else:
    logger.info("  ⏭️  Network analysis disabled in config")


# --- Step 5/7: Research Gap Detection (Step 022) ---
logger.info("\n[Analysis Step 2/2] Research gap detection...")

gap_analysis_results: Dict[str, Any] = {}

if orchestration_config.enable_gap_analysis:
    try:
        # Execute gap analysis step with retry logic
        gap_result = retry_step(step_022_gap_analysis, orchestration_config)
        analysis_step_results['gap_analysis'] = gap_result
        
        if gap_result.success:
            # Extract gap metrics from artifacts
            gaps_identified = gap_result.artifacts.get('gaps_identified', 0)
            rqs_analyzed = gap_result.artifacts.get('rqs_analyzed', 0)
            
            logger.info(f"  ✅ Gap analysis completed")
            logger.info(f"     Research questions analyzed: {rqs_analyzed}")
            logger.info(f"     Gaps identified: {gaps_identified}")
            logger.info(f"     Duration: {gap_result.duration_seconds:.1f}s")
            
            # TODO: In actual implementation, this step would:
            # - Query Notion RQ_DB for all active research questions
            # - For each RQ:
            #   * Count related literature (via relation property)
            #   * Extract themes and methods from RQ and related papers
            #   * Compute coverage score (evidence count vs expected)
            # - Identify under-covered RQs (low evidence count, high priority)
            # - Identify theme gaps (themes in RQs but not in literature)
            # - Identify method gaps (methods needed but not applied)
            # - Use LLM to synthesize gap narratives:
            #   * Prompt: "Given RQ X with Y papers on themes Z, what gaps exist?"
            #   * Generate 2-3 sentence gap description per under-covered RQ
            # - Prioritize gaps by RQ priority and coverage deficit
            # - Save gap analysis to checkpoint with LLM-generated narratives
            
            gap_analysis_results = gap_result.artifacts.copy()
            
            # TODO: Load detailed gap data from checkpoint
            # checkpoint_data = load_checkpoint(gap_result.checkpoint_path)
            # gap_analysis_results['gap_details'] = checkpoint_data.get('gap_details', [])
            
            # Placeholder: simulate gap structure
            gap_analysis_results.update({
                'under_covered_rqs': [
                    {
                        'rq_id': f'rq_{i+1}',
                        'question': f'[Research Question {i+1}]',
                        'evidence_count': i,
                        'expected_count': 10 + i*2,
                        'coverage_deficit': 10 + i,
                        'gap_narrative': f'[LLM-generated gap description for RQ {i+1}]'
                    }
                    for i in range(gaps_identified)
                ],
                'theme_gaps': [
                    {'theme': f'[Underexplored theme {i+1}]', 'rq_count': 3 - i}
                    for i in range(min(3, gaps_identified))
                ],
                'method_gaps': [
                    {'method': f'[Needed method {i+1}]', 'rq_count': 2}
                    for i in range(min(2, gaps_identified))
                ],
            })
            
            if VERBOSE:
                logger.debug(f"\nGap analysis summary:")
                logger.debug(json.dumps(gap_analysis_results, indent=2, default=str))
                
            # Log top gaps
            if gap_analysis_results.get('under_covered_rqs'):
                logger.info(f"\n  Top under-covered research questions:")
                for rq in gap_analysis_results['under_covered_rqs'][:3]:
                    logger.info(f"    - {rq['question'][:60]}...")
                    logger.info(f"      Evidence: {rq['evidence_count']}/{rq['expected_count']} (deficit: {rq['coverage_deficit']})")
                    
        else:
            # Step failed
            error_msg = gap_result.error or 'Unknown error in gap analysis'
            logger.error(f"  ❌ Gap analysis failed: {error_msg}")
            analysis_failures.append(
                PipelineFailure(
                    step_name='gap_analysis',
                    error_message=error_msg,
                    timestamp=datetime.now()
                )
            )
            
    except Exception as e:
        error_msg = f"Exception during gap analysis: {str(e)}"
        logger.error(f"  ❌ {error_msg}")
        logger.debug(traceback.format_exc())
        
        analysis_failures.append(
            PipelineFailure(
                step_name='gap_analysis',
                error_message=error_msg,
                timestamp=datetime.now(),
                stack_trace=traceback.format_exc()
            )
        )
else:
    logger.info("  ⏭️  Gap analysis disabled in config")


# --- Analysis workflow summary ---
analysis_duration = (datetime.now() - analysis_start_time).total_seconds()

logger.info("\n" + "="*60)
logger.info("ANALYSIS WORKFLOW COMPLETED")
logger.info(f"Total duration: {analysis_duration:.1f}s")
logger.info(f"Steps executed: {len(analysis_step_results)}/2")
logger.info(f"Failures: {len(analysis_failures)}")

if network_metrics:
    logger.info(f"Network nodes analyzed: {network_metrics.get('nodes', 0)}")
if gap_analysis_results:
    logger.info(f"Research gaps identified: {gap_analysis_results.get('gaps_identified', 0)}")

if analysis_failures:
    logger.warning("\nFailures encountered:")
    for failure in analysis_failures:
        logger.warning(f"  - {failure.step_name}: {failure.error_message}")
else:
    logger.info("\n✅ All analysis steps completed successfully")

logger.info("="*60)

# Store results for downstream use in cells 13-14
analysis_workflow_summary = {
    'step_results': analysis_step_results,
    'failures': [asdict(f) for f in analysis_failures],
    'duration_seconds': analysis_duration,
    'network_metrics': network_metrics,
    'gap_analysis': gap_analysis_results,
    'success': len(analysis_failures) == 0,
}

if VERBOSE:
    logger.debug("\nAnalysis workflow summary:")
    logger.debug(json.dumps(analysis_workflow_summary, indent=2, default=str))


2026-01-20 10:49:57 [INFO] orchestrator: 
2026-01-20 10:49:57 [INFO] orchestrator: ANALYSIS WORKFLOW (Steps 021-022)
2026-01-20 10:49:57 [INFO] orchestrator: ============================================================
2026-01-20 10:49:57 [INFO] orchestrator: 
[Analysis Step 1/2] Citation network analysis...
2026-01-20 10:49:57 [INFO] orchestrator:   ✅ Network analysis completed
2026-01-20 10:49:57 [INFO] orchestrator:      Network nodes: 0
2026-01-20 10:49:57 [INFO] orchestrator:      Citation edges: 0
2026-01-20 10:49:57 [INFO] orchestrator:      Identified clusters: 0
2026-01-20 10:49:57 [INFO] orchestrator:      Duration: 0.0s
2026-01-20 10:49:57 [INFO] orchestrator: 
[Analysis Step 2/2] Research gap detection...
2026-01-20 10:49:57 [INFO] orchestrator:   ✅ Gap analysis completed
2026-01-20 10:49:57 [INFO] orchestrator:      Research questions analyzed: 0
2026-01-20 10:49:57 [INFO] orchestrator:      Gaps identified: 0
2026-01-20 10:49:57 [INFO] orchestrator:      Duration: 0.0s
20

In [30]:
# ============================================================
# Cell 12 — Execute daily scanning and RQ update steps (023-024)
# ============================================================
# Overview:
#   Daily workflow:
#     (023) Fetch new papers (arXiv, last N days), filter by simple relevance,
#          optionally ingest to Notion LIT_DB, and generate X draft markdown.
#     (024) Fetch recent meetings from MTG_DB and active RQs from RQ_DB,
#          then generate RQ update proposals (markdown) via LLM.
#
# Inputs / Outputs:
#   Inputs:
#     - orchestration_config, env_config
#     - openai_client
#     - notion_rest (requests-based client from Cell 04)
#     - LIT_SCHEMA / MEETING_SCHEMA / RQ_SCHEMA, SCHEMA_CONFIG
#     - REPORT_FILES
#   Outputs:
#     - Writes reports: x_draft.md, rq_update_proposal.md
#     - Optionally creates new pages in Notion LIT_DB (if notion_write_enabled)
#
# Notes:
#   - Safe-by-default: does NOT mutate existing RQ pages; outputs proposals to md.
#   - Uses simple keyword relevance; you can replace with embedding/LLM classifier later.
#   - arXiv API is Atom feed; we parse via xml.etree.
#

import re
import time
import requests
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
from typing import Dict, Any, List, Optional, Tuple

# -----------------------------
# Config knobs (edit here)
# -----------------------------
ARXIV_DAYS_BACK = 3              # "daily" window (1 = last 1 day approx)
ARXIV_MAX_RESULTS = min(orchestration_config.max_papers_per_run, 50)
ARXIV_SORT_BY = "submittedDate"  # submittedDate | lastUpdatedDate | relevance

# Put your daily keywords here (quick win). Later, auto-build from RQs.
DAILY_KEYWORDS = [
    "startup", "venture capital", "innovation", "entrepreneurship",
    "AI", "foundation model", "LLM", "science of science", "policy",
    "research automation", "knowledge graph", "citation network"
]

# If you already have a canonical query string, set it here.
# arXiv query syntax docs: http://arxiv.org/help/api/user-manual
ARXIV_QUERY = (
    'all:"startup" OR all:"venture capital" OR all:"innovation policy" '
    'OR all:"science of science" OR all:"citation network" OR all:"knowledge graph" '
    'OR all:"large language model" OR all:"foundation model"'
)

# Meeting lookback for RQ update proposals
MEETING_DAYS_BACK = 14

# RQ selection: adjust according to your RQ DB Status values
RQ_STATUS_ALLOWLIST = ["Active"]  # if your DB uses different labels, edit here

# -----------------------------
# Helpers: Notion property builders (mapping-driven)
# -----------------------------
def notion_prop_title(text: str) -> Dict[str, Any]:
    return {"title": [{"type": "text", "text": {"content": text[:2000]}}]}

def notion_prop_rich_text(text: str) -> Dict[str, Any]:
    if not text:
        return {"rich_text": []}
    return {"rich_text": [{"type": "text", "text": {"content": text[:2000]}}]}

def notion_prop_url(url: str) -> Dict[str, Any]:
    return {"url": url} if url else {"url": None}

def notion_prop_select(name: str) -> Dict[str, Any]:
    return {"select": {"name": name}} if name else {"select": None}

def notion_prop_multi_select(names: List[str]) -> Dict[str, Any]:
    names = [n for n in (names or []) if n]
    return {"multi_select": [{"name": n[:100]} for n in names[:20]]}

# -----------------------------
# Helpers: Notion REST query (light wrapper)
# -----------------------------
def notion_query_all(database_id: str, payload: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Fetch all pages with pagination."""
    results: List[Dict[str, Any]] = []
    cursor = None
    while True:
        body = dict(payload or {})
        if cursor:
            body["start_cursor"] = cursor
        data = notion_rest.query_database(database_id, body)
        results.extend(data.get("results", []))
        if not data.get("has_more"):
            break
        cursor = data.get("next_cursor")
        if not cursor:
            break
        time.sleep(0.2)
    return results

def extract_plain_text_rich(prop: Dict[str, Any]) -> str:
    """Extract plain text from rich_text or title property dict."""
    if not prop:
        return ""
    if "rich_text" in prop:
        parts = prop.get("rich_text") or []
        return "".join([p.get("plain_text", "") for p in parts]).strip()
    if "title" in prop:
        parts = prop.get("title") or []
        return "".join([p.get("plain_text", "") for p in parts]).strip()
    return ""

# -----------------------------
# Step 023: arXiv fetch + relevance filter
# -----------------------------
def arxiv_fetch(query: str, max_results: int, sort_by: str = "submittedDate") -> List[Dict[str, Any]]:
    base = "https://export.arxiv.org/api/query"
    params = {
        "search_query": query,
        "start": 0,
        "max_results": int(max_results),
        "sortBy": sort_by,
        "sortOrder": "descending",
    }
    r = requests.get(base, params=params, timeout=60)
    r.raise_for_status()

    root = ET.fromstring(r.text)
    ns = {"atom": "http://www.w3.org/2005/Atom"}

    papers: List[Dict[str, Any]] = []
    for entry in root.findall("atom:entry", ns):
        title = (entry.findtext("atom:title", default="", namespaces=ns) or "").strip()
        summary = (entry.findtext("atom:summary", default="", namespaces=ns) or "").strip()
        published = (entry.findtext("atom:published", default="", namespaces=ns) or "").strip()
        updated = (entry.findtext("atom:updated", default="", namespaces=ns) or "").strip()

        authors = []
        for a in entry.findall("atom:author", ns):
            name = a.findtext("atom:name", default="", namespaces=ns)
            if name:
                authors.append(name.strip())

        # id + links
        arxiv_id_url = (entry.findtext("atom:id", default="", namespaces=ns) or "").strip()
        pdf_url = ""
        for link in entry.findall("atom:link", ns):
            if link.get("type") == "application/pdf":
                pdf_url = link.get("href") or ""
                break

        papers.append({
            "title": title,
            "abstract": summary,
            "authors": authors,
            "published": published,
            "updated": updated,
            "url": arxiv_id_url,
            "pdf_url": pdf_url,
            "source": "arXiv",
            "arxiv_id": arxiv_id_url.rsplit("/", 1)[-1] if arxiv_id_url else None,
        })

    return papers

def within_days(iso_dt: str, days_back: int) -> bool:
    """arXiv published is ISO8601; compare with now."""
    try:
        dt = datetime.fromisoformat(iso_dt.replace("Z", "+00:00"))
    except Exception:
        return True  # if parse fails, don't filter out
    return dt >= (datetime.now(dt.tzinfo) - timedelta(days=days_back))

def simple_relevance_score(text: str, keywords: List[str]) -> Tuple[int, List[str]]:
    text_l = (text or "").lower()
    hits = []
    for kw in keywords:
        if kw.lower() in text_l:
            hits.append(kw)
    return len(hits), hits

def step_023_daily_scan_real(config: OrchestrationConfig) -> Tuple[List[Dict[str, Any]], str]:
    """Return (relevant_papers, x_draft_markdown)."""
    logger.info("[Daily Step 1/2] Fetching new papers from arXiv...")

    all_papers = arxiv_fetch(ARXIV_QUERY, ARXIV_MAX_RESULTS, sort_by=ARXIV_SORT_BY)
    recent = [p for p in all_papers if within_days(p.get("published", ""), ARXIV_DAYS_BACK)]
    logger.info(f"  arXiv fetched: {len(all_papers)} / recent({ARXIV_DAYS_BACK}d): {len(recent)}")

    # relevance
    relevant: List[Dict[str, Any]] = []
    for p in recent:
        text = f"{p.get('title','')} {p.get('abstract','')}"
        score, hits = simple_relevance_score(text, DAILY_KEYWORDS)
        if score >= 1:
            p["relevance_score"] = score
            p["keyword_hits"] = hits
            relevant.append(p)

    # sort by score desc, then published desc
    relevant.sort(key=lambda x: (x.get("relevance_score", 0), x.get("published", "")), reverse=True)

    logger.info(f"  Relevant papers: {len(relevant)} (keyword hit >= 1)")

    # X draft (top 3)
    top = relevant[:3]
    if top:
        bullets = []
        for i, p in enumerate(top, 1):
            title = p["title"]
            url = p.get("url", "")
            bullets.append(f"{i}) {title}\n   {url}")
        x_md = (
            "## X Draft (Daily Paper Highlights)\n\n"
            "Draft:\n\n"
            + "New papers worth a look today:\n\n"
            + "\n\n".join(bullets)
            + "\n\n#researchOS100"
        )
    else:
        x_md = (
            "## X Draft (Daily Paper Highlights)\n\n"
            "No relevant new papers were found in the last window.\n"
            "Consider widening the query or increasing days_back.\n\n"
            "#researchOS100"
        )

    return relevant, x_md

# -----------------------------
# Optional: ingest relevant papers to Notion LIT_DB (safe minimal create)
# -----------------------------
def notion_create_lit_page(paper: Dict[str, Any]) -> Dict[str, Any]:
    """Create a page in LIT_DB using mapping keys; minimal fields only."""
    db_id = env_config.lit_db
    props = {}

    # Title
    props[LIT_SCHEMA["title"]] = notion_prop_title(paper.get("title", "Untitled"))

    # Authors & Year (as rich text)
    authors = paper.get("authors", [])
    year = ""
    try:
        year = (paper.get("published", "") or "")[:4]
    except Exception:
        year = ""
    authors_year = ", ".join(authors[:10]) + (f" ({year})" if year else "")
    if "authors_year" in LIT_SCHEMA:
        props[LIT_SCHEMA["authors_year"]] = notion_prop_rich_text(authors_year)

    # Source (select)
    if "source" in LIT_SCHEMA:
        props[LIT_SCHEMA["source"]] = notion_prop_select(paper.get("source", "arXiv"))

    # PDF Link (url)
    if "pdf_link" in LIT_SCHEMA:
        props[LIT_SCHEMA["pdf_link"]] = notion_prop_url(paper.get("pdf_url", "") or paper.get("url", ""))

    # Tags (multi_select) - use keyword hits (optional)
    if "tags" in LIT_SCHEMA:
        hits = paper.get("keyword_hits", [])
        props[LIT_SCHEMA["tags"]] = notion_prop_multi_select(hits[:10])

    payload = {
        "parent": {"database_id": db_id},
        "properties": props,
    }

    url = "https://api.notion.com/v1/pages"
    r = requests.post(url, headers=NOTION_HEADERS, json=payload, timeout=60)
    if r.status_code >= 400:
        try:
            detail = r.json()
        except Exception:
            detail = {"text": r.text}
        raise RuntimeError(f"Notion create page failed: HTTP {r.status_code} - {detail}")
    return r.json()

def ingest_papers_to_notion_lit(papers: List[Dict[str, Any]], batch_size: int = 10) -> Dict[str, int]:
    created = 0
    failed = 0

    # NOTE: minimal duplicate prevention (by arXiv ID) can be added later;
    # for now we do a lightweight title-based check against recent entries.
    # (Keeps this cell minimal and reliable.)
    for i in range(0, len(papers), batch_size):
        chunk = papers[i:i+batch_size]
        for p in chunk:
            try:
                notion_create_lit_page(p)
                created += 1
            except Exception as e:
                failed += 1
                logger.warning(f"  Notion ingest failed for '{p.get('title','')[:60]}...': {e}")
        time.sleep(0.3)  # be gentle to API
    return {"created": created, "failed": failed}

# -----------------------------
# Step 024: RQ update proposals from meetings + new papers (LLM)
# -----------------------------
def fetch_recent_meetings(days_back: int) -> List[Dict[str, Any]]:
    db_id = env_config.mtg_db
    since = (datetime.now() - timedelta(days=days_back)).date().isoformat()

    # Filter: Date on/after since (if your Date prop is correct)
    date_prop = MEETING_SCHEMA["date"]
    payload = {
        "filter": {
            "property": date_prop,
            "date": {"on_or_after": since}
        },
        "sorts": [{"property": date_prop, "direction": "descending"}],
        "page_size": 50
    }
    return notion_query_all(db_id, payload)

def fetch_active_rqs() -> List[Dict[str, Any]]:
    db_id = env_config.rq_db
    status_prop_name = RQ_SCHEMA["status"]

    # --- Detect property type from DB schema (REST) ---
    db_info = notion_rest.retrieve_database(db_id)
    props = db_info.get("properties", {}) or {}

    if status_prop_name not in props:
        raise RuntimeError(f"RQ_DB is missing status property: '{status_prop_name}'. Available: {list(props.keys())[:20]}")

    status_prop_def = props[status_prop_name]
    status_prop_type = status_prop_def.get("type")  # "select" or "status" etc.

    # Build OR filter depending on property type
    if status_prop_type == "status":
        ors = [{"property": status_prop_name, "status": {"equals": s}} for s in RQ_STATUS_ALLOWLIST]
    elif status_prop_type == "select":
        ors = [{"property": status_prop_name, "select": {"equals": s}} for s in RQ_STATUS_ALLOWLIST]
    else:
        # Fall back: no filter (still returns something)
        logger.warning(
            f"⚠️  RQ_DB status property type is '{status_prop_type}', not handled. "
            "Fetching without status filter."
        )
        ors = []

    payload = {"page_size": 50}
    if ors:
        payload["filter"] = {"or": ors}

    return notion_query_all(db_id, payload)


def summarize_meetings_for_llm(meetings: List[Dict[str, Any]]) -> str:
    lines = []
    for m in meetings[:10]:
        props = m.get("properties", {})
        title = extract_plain_text_rich(props.get(MEETING_SCHEMA["title"], {})) or "Untitled"
        date = ""
        try:
            date_obj = props.get(MEETING_SCHEMA["date"], {}).get("date")
            date = (date_obj or {}).get("start", "") or ""
        except Exception:
            date = ""
        summary = extract_plain_text_rich(props.get(MEETING_SCHEMA["notes_primary"], {}))
        if not summary:
            summary = extract_plain_text_rich(props.get(MEETING_SCHEMA["notes_fallback"], {}))
        summary = (summary or "")[:800]
        lines.append(f"- {date} | {title}\n  {summary}")
    return "\n".join(lines).strip() or "(No recent meetings found.)"

def summarize_rqs_for_llm(rqs: List[Dict[str, Any]]) -> str:
    lines = []
    for rq in rqs[:20]:
        props = rq.get("properties", {})
        title = extract_plain_text_rich(props.get(RQ_SCHEMA["title"], {})) or "Untitled RQ"
        pr = props.get(RQ_SCHEMA["priority"], {}).get("select", {}) or {}
        pr_name = pr.get("name", "")
        rationale = extract_plain_text_rich(props.get(RQ_SCHEMA["rationale"], {}))[:400]
        gap = extract_plain_text_rich(props.get(RQ_SCHEMA["gap"], {}))[:300]
        lines.append(f"- {title} (Priority: {pr_name})\n  Rationale: {rationale}\n  Gap: {gap}")
    return "\n".join(lines).strip() or "(No active RQs found.)"

def summarize_new_papers_for_llm(papers: List[Dict[str, Any]]) -> str:
    if not papers:
        return "(No new relevant papers found.)"
    lines = []
    for p in papers[:10]:
        lines.append(f"- {p['title']}\n  {p.get('url','')}")
    return "\n".join(lines)

def generate_rq_update_proposal_md(meetings_text: str, rqs_text: str, papers_text: str) -> str:
    prompt = f"""
You are a PhD research assistant. Based on:
1) Recent meeting notes
2) Current active research questions (RQs)
3) Newly found relevant papers

Propose:
- Up to 5 concrete RQ updates (refine wording, scope, priorities, or sub-questions)
- For each update: rationale and what evidence triggered it (meeting and/or paper)
- If nothing should change, say so and explain why.

Return in Markdown with headings and bullet points. Be concise but specific.

[Recent meetings]
{meetings_text}

[Active RQs]
{rqs_text}

[New papers]
{papers_text}
""".strip()

    resp = openai_client.chat.completions.create(
        model=llm_model,
        temperature=float(llm_temperature) if llm_temperature is not None else 0.0,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=900,
    )
    content = resp.choices[0].message.content.strip()
    return "## RQ Update Proposal (Daily)\n\n" + content

# -----------------------------
# Execute Daily Workflow
# -----------------------------
logger.info("\n" + "="*60)
logger.info("DAILY WORKFLOW (Steps 023-024)")
logger.info("="*60)

daily_start = datetime.now()

# Step 023: Daily scan
relevant_papers, x_md = step_023_daily_scan_real(orchestration_config)

# Optionally ingest to Notion LIT_DB
ingest_stats = {"created": 0, "failed": 0}
if relevant_papers and orchestration_config.notion_write_enabled:
    logger.info(f"[Daily Step 1/2] Ingesting {len(relevant_papers)} relevant papers to Notion LIT_DB...")
    ingest_stats = ingest_papers_to_notion_lit(relevant_papers[:20], batch_size=orchestration_config.notion_batch_size)
    logger.info(f"  Notion ingest: created={ingest_stats['created']}, failed={ingest_stats['failed']}")
elif relevant_papers and not orchestration_config.notion_write_enabled:
    logger.warning("[Daily Step 1/2] Notion writes disabled (dry-run). Skipping LIT_DB ingestion.")
else:
    logger.info("[Daily Step 1/2] No relevant papers found. Skipping LIT_DB ingestion.")

# Write X draft report
x_path = REPORT_FILES["x_draft"]
x_path.write_text(x_md + "\n", encoding="utf-8")
logger.info(f"✅ Wrote X draft: {x_path} ({x_path.stat().st_size/1024:.1f} KB)")

# Step 024: RQ update proposal (meetings + RQs + papers)
logger.info("[Daily Step 2/2] Generating RQ update proposal from meetings + new papers...")

recent_meetings = fetch_recent_meetings(MEETING_DAYS_BACK)
active_rqs = fetch_active_rqs()

meetings_text = summarize_meetings_for_llm(recent_meetings)
rqs_text = summarize_rqs_for_llm(active_rqs)
papers_text = summarize_new_papers_for_llm(relevant_papers)

rq_md = generate_rq_update_proposal_md(meetings_text, rqs_text, papers_text)

# Add a small execution footer
rq_md += (
    "\n\n---\n"
    f"- Run ID: {orchestration_config.run_id}\n"
    f"- Window: arXiv last ~{ARXIV_DAYS_BACK} day(s), meetings last {MEETING_DAYS_BACK} day(s)\n"
    f"- Relevant papers: {len(relevant_papers)} (ingested: {ingest_stats['created']})\n"
    f"- Generated: {datetime.now().isoformat(timespec='seconds')}\n"
)

rq_path = REPORT_FILES["rq_update_proposal"]
rq_path.write_text(rq_md + "\n", encoding="utf-8")
logger.info(f"✅ Wrote RQ proposal: {rq_path} ({rq_path.stat().st_size/1024:.1f} KB)")

daily_dur = (datetime.now() - daily_start).total_seconds()
logger.info("="*60)
logger.info(f"DAILY WORKFLOW DONE in {daily_dur:.1f}s")
logger.info(f"- relevant_papers: {len(relevant_papers)}")
logger.info(f"- LIT_DB ingested: created={ingest_stats['created']} failed={ingest_stats['failed']}")
logger.info("="*60)


2026-01-20 11:03:59 [INFO] orchestrator: 
2026-01-20 11:03:59 [INFO] orchestrator: DAILY WORKFLOW (Steps 023-024)
2026-01-20 11:03:59 [INFO] orchestrator: ============================================================
2026-01-20 11:03:59 [INFO] orchestrator: [Daily Step 1/2] Fetching new papers from arXiv...
2026-01-20 11:03:59 [INFO] orchestrator:   arXiv fetched: 50 / recent(3d): 0
2026-01-20 11:03:59 [INFO] orchestrator:   Relevant papers: 0 (keyword hit >= 1)
2026-01-20 11:03:59 [INFO] orchestrator: [Daily Step 1/2] No relevant papers found. Skipping LIT_DB ingestion.
2026-01-20 11:03:59 [INFO] orchestrator: ✅ Wrote X draft: runs/2026-01-20_1044/reports/x_draft.md (0.2 KB)
2026-01-20 11:03:59 [INFO] orchestrator: [Daily Step 2/2] Generating RQ update proposal from meetings + new papers...
2026-01-20 11:04:20 [INFO] httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-20 11:04:20 [INFO] orchestrator: ✅ Wrote RQ proposal: runs/2026-01-20_1044/r

In [31]:
# ============================================================
# Cell 13 — Compile final reports and write to run directory
# ============================================================
# Overview:
#   Aggregate results from all workflow steps (weekly corpus, analysis, daily)
#   and compile five markdown reports using functions from cell 08. Write all
#   reports to the run directory's reports/ subdirectory.
#
# Inputs / Outputs:
#   Inputs: weekly_corpus_summary, analysis_workflow_summary, daily_workflow_summary
#   Outputs: Five markdown reports written to orchestration_config.reports_dir
#
# Notes:
#   - Combines step_results from all three workflows into single dict
#   - Calls report compilation functions from cell 08
#   - Writes reports only if corresponding config flags are enabled
#   - Logs success/failure for each report
#   - Safe to run even if some workflows failed (graceful degradation)
#

# --- Aggregate all step results from workflows ---
logger.info("\n" + "="*60)
logger.info("COMPILING FINAL REPORTS")
logger.info("="*60)

# Combine step results from all three workflows
all_step_results: Dict[str, StepResult] = {}
all_step_results.update(weekly_corpus_summary.get('step_results', {}))
all_step_results.update(analysis_workflow_summary.get('step_results', {}))
all_step_results.update(daily_workflow_summary.get('step_results', {}))

logger.info(f"Aggregated results from {len(all_step_results)} pipeline steps")

# Aggregate all failures
all_failures = (
    weekly_corpus_summary.get('failures', []) +
    analysis_workflow_summary.get('failures', []) +
    daily_workflow_summary.get('failures', [])
)

if all_failures:
    logger.warning(f"Total failures across all workflows: {len(all_failures)}")

# --- Compile and write reports ---
report_write_status = {}
reports_written = 0

try:
    # Use write_all_reports function from cell 08
    logger.info("\nGenerating report files...")
    report_write_status = write_all_reports(all_step_results, orchestration_config)
    
    reports_written = sum(1 for status in report_write_status.values() if status)
    
    logger.info(f"\n✅ Successfully wrote {reports_written}/5 reports")
    
    # Log individual report paths
    for report_name, written in report_write_status.items():
        if written:
            path = REPORT_FILES[report_name]
            logger.info(f"   📄 {path.name}")
            if VERBOSE:
                logger.debug(f"      Full path: {path}")
    
except Exception as e:
    logger.error(f"❌ Error during report compilation: {e}")
    if VERBOSE:
        logger.debug(traceback.format_exc())
    report_write_status = {'error': str(e)}

logger.info("="*60)

# Store report status for manifest (cell 14)
report_compilation_summary = {
    'reports_written': reports_written,
    'report_status': report_write_status,
    'total_steps_aggregated': len(all_step_results),
    'total_failures': len(all_failures),
}


2026-01-20 11:04:26 [INFO] orchestrator: 
2026-01-20 11:04:26 [INFO] orchestrator: COMPILING FINAL REPORTS
2026-01-20 11:04:26 [INFO] orchestrator: ============================================================
2026-01-20 11:04:26 [INFO] orchestrator: Aggregated results from 5 pipeline steps
2026-01-20 11:04:26 [INFO] orchestrator: 
Generating report files...
2026-01-20 11:04:26 [INFO] orchestrator: Compiling final reports...
2026-01-20 11:04:26 [INFO] orchestrator:   ✅ weekly_corpus_update_report.md
2026-01-20 11:04:26 [INFO] orchestrator:   ✅ network_update_summary.md
2026-01-20 11:04:26 [INFO] orchestrator:   ✅ gap_update.md
2026-01-20 11:04:26 [INFO] orchestrator:   ✅ rq_update_proposal.md
2026-01-20 11:04:26 [INFO] orchestrator:   ✅ x_draft.md
2026-01-20 11:04:26 [INFO] orchestrator: 
✅ Successfully wrote 5/5 reports
2026-01-20 11:04:26 [INFO] orchestrator:    📄 weekly_corpus_update_report.md
2026-01-20 11:04:26 [INFO] orchestrator:    📄 network_update_summary.md
2026-01-20 11:04:26

In [32]:
# ============================================================
# Cell 14 — Generate run manifest and summary statistics
# ============================================================
# Overview:
#   Generate comprehensive run manifest JSON file containing all pipeline
#   execution metadata, step results, failures, and summary statistics.
#   Provides deterministic record of run for auditing, debugging, and
#   historical analysis. Includes environment config snapshot, timing metrics,
#   and output file inventory.
#
# Inputs / Outputs:
#   Inputs: orchestration_config, all workflow summaries, report_compilation_summary
#   Outputs: run_manifest.json written to run directory root
#
# Notes:
#   - Manifest is complete standalone record of pipeline execution
#   - Safe for serialization (no sensitive credentials included)
#   - Uses ISO timestamps for all time fields
#   - Includes success/failure flags for each step and overall run
#   - Can be loaded for post-run analysis or CI/CD integration
#

# --- Collect all execution metadata ---
logger.info("\n" + "="*60)
logger.info("GENERATING RUN MANIFEST")
logger.info("="*60)

manifest_start_time = datetime.now()

# --- Build manifest structure ---
run_manifest = {
    # Run identification
    'run_id': orchestration_config.run_id,
    'run_timestamp': orchestration_config.run_timestamp.isoformat(),
    'manifest_generated': manifest_start_time.isoformat(),
    
    # Environment snapshot
    'environment': {
        'dry_run': orchestration_config.dry_run,
        'llm_provider': llm_provider,
        'llm_model': llm_model,
        'llm_temperature': llm_temperature,
        'env_config': env_config.to_dict(),
        'validation_summary': validation_summary,
    },
    
    # Orchestration configuration
    'orchestration_config': orchestration_config.to_dict(),
    
    # Step execution results
    'steps': {},
    
    # Workflow summaries
    'workflows': {
        'weekly_corpus_update': {
            'duration_seconds': weekly_corpus_summary.get('duration_seconds', 0.0),
            'success': weekly_corpus_summary.get('success', False),
            'papers_discovered': weekly_corpus_summary.get('papers_discovered', 0),
            'failures': weekly_corpus_summary.get('failures', []),
        },
        'analysis': {
            'duration_seconds': analysis_workflow_summary.get('duration_seconds', 0.0),
            'success': analysis_workflow_summary.get('success', False),
            'network_metrics': {
                'nodes': analysis_workflow_summary.get('network_metrics', {}).get('nodes', 0),
                'edges': analysis_workflow_summary.get('network_metrics', {}).get('edges', 0),
                'clusters': analysis_workflow_summary.get('network_metrics', {}).get('clusters', 0),
            },
            'gaps_identified': analysis_workflow_summary.get('gap_analysis', {}).get('gaps_identified', 0),
            'failures': analysis_workflow_summary.get('failures', []),
        },
        'daily_maintenance': {
            'duration_seconds': daily_workflow_summary.get('duration_seconds', 0.0),
            'success': daily_workflow_summary.get('success', False),
            'new_papers_added': daily_workflow_summary.get('new_papers_added', 0),
            'rqs_updated': daily_workflow_summary.get('rqs_updated', 0),
            'failures': daily_workflow_summary.get('failures', []),
        },
    },
    
    # Report generation
    'reports': {
        'reports_written': report_compilation_summary.get('reports_written', 0),
        'report_status': report_compilation_summary.get('report_status', {}),
        'report_files': {name: str(path) for name, path in REPORT_FILES.items()},
    },
    
    # Overall run summary
    'summary': {},
    
    # Checkpoint files inventory
    'checkpoints': {name: str(path) for name, path in CHECKPOINT_FILES.items()},
    
    # Output file inventory
    'output_files': {},
}

# --- Populate step-level details ---
logger.info("Aggregating step execution details...")

for step_name, step_result in all_step_results.items():
    run_manifest['steps'][step_name] = {
        'success': step_result.success,
        'duration_seconds': step_result.duration_seconds,
        'artifacts': step_result.artifacts,
        'error': step_result.error,
        'checkpoint_path': str(step_result.checkpoint_path) if step_result.checkpoint_path else None,
    }

logger.info(f"  Captured details for {len(all_step_results)} steps")

# --- Calculate summary statistics ---
logger.info("Computing summary statistics...")

# Total duration across all workflows
total_duration = (
    weekly_corpus_summary.get('duration_seconds', 0.0) +
    analysis_workflow_summary.get('duration_seconds', 0.0) +
    daily_workflow_summary.get('duration_seconds', 0.0)
)

# Success metrics
steps_executed = len(all_step_results)
steps_succeeded = sum(1 for r in all_step_results.values() if r.success)
steps_failed = steps_executed - steps_succeeded

# Overall success flag (all workflows succeeded)
overall_success = (
    weekly_corpus_summary.get('success', False) and
    analysis_workflow_summary.get('success', False) and
    daily_workflow_summary.get('success', False)
)

# Papers processed
papers_discovered = weekly_corpus_summary.get('papers_discovered', 0)
new_papers_daily = daily_workflow_summary.get('new_papers_added', 0)
total_papers_processed = papers_discovered + new_papers_daily

# Analysis outputs
network_nodes = analysis_workflow_summary.get('network_metrics', {}).get('nodes', 0)
gaps_identified = analysis_workflow_summary.get('gap_analysis', {}).get('gaps_identified', 0)
rqs_updated = daily_workflow_summary.get('rqs_updated', 0)

# Build summary dict
run_manifest['summary'] = {
    'overall_success': overall_success,
    'total_duration_seconds': total_duration,
    'steps_executed': steps_executed,
    'steps_succeeded': steps_succeeded,
    'steps_failed': steps_failed,
    'total_failures': len(all_failures),
    'papers_processed': {
        'weekly_corpus': papers_discovered,
        'daily_scan': new_papers_daily,
        'total': total_papers_processed,
    },
    'analysis_outputs': {
        'network_nodes': network_nodes,
        'gaps_identified': gaps_identified,
        'rqs_updated': rqs_updated,
    },
    'reports_generated': report_compilation_summary.get('reports_written', 0),
}

logger.info("  Summary statistics computed")

# --- Enumerate output files ---
logger.info("Enumerating output files...")

output_files = {
    'reports': [],
    'checkpoints': [],
    'logs': [],
    'artifacts': [],
}

# Scan reports directory
if orchestration_config.reports_dir.exists():
    for file_path in orchestration_config.reports_dir.iterdir():
        if file_path.is_file():
            output_files['reports'].append({
                'name': file_path.name,
                'path': str(file_path),
                'size_bytes': file_path.stat().st_size,
            })

# Scan artifacts directory
if orchestration_config.artifacts_dir.exists():
    for file_path in orchestration_config.artifacts_dir.iterdir():
        if file_path.is_file():
            file_info = {
                'name': file_path.name,
                'path': str(file_path),
                'size_bytes': file_path.stat().st_size,
            }
            if '_checkpoint.json' in file_path.name:
                output_files['checkpoints'].append(file_info)
            else:
                output_files['artifacts'].append(file_info)

# Scan logs directory
if orchestration_config.logs_dir.exists():
    for file_path in orchestration_config.logs_dir.iterdir():
        if file_path.is_file():
            output_files['logs'].append({
                'name': file_path.name,
                'path': str(file_path),
                'size_bytes': file_path.stat().st_size,
            })

run_manifest['output_files'] = output_files

logger.info(f"  Found {len(output_files['reports'])} report files")
logger.info(f"  Found {len(output_files['checkpoints'])} checkpoint files")
logger.info(f"  Found {len(output_files['logs'])} log files")
logger.info(f"  Found {len(output_files['artifacts'])} artifact files")

# --- Write manifest to file ---
manifest_path = orchestration_config.run_dir / 'run_manifest.json'

logger.info(f"\nWriting manifest to {manifest_path.name}...")

try:
    with open(manifest_path, 'w') as f:
        json.dump(run_manifest, f, indent=2, default=str)
    
    manifest_size = manifest_path.stat().st_size
    logger.info(f"  ✅ Manifest written successfully ({manifest_size:,} bytes)")
    
except Exception as e:
    logger.error(f"  ❌ Failed to write manifest: {e}")
    if VERBOSE:
        logger.debug(traceback.format_exc())

# --- Print summary to console ---
logger.info("\n" + "="*60)
logger.info("RUN SUMMARY")
logger.info("="*60)

logger.info(f"\n📊 Overall Status: {'✅ SUCCESS' if overall_success else '❌ FAILED'}")
logger.info(f"\n⏱️  Execution Time:")
logger.info(f"   Total duration: {total_duration:.1f}s ({total_duration/60:.1f} minutes)")
logger.info(f"   Weekly corpus: {weekly_corpus_summary.get('duration_seconds', 0.0):.1f}s")
logger.info(f"   Analysis: {analysis_workflow_summary.get('duration_seconds', 0.0):.1f}s")
logger.info(f"   Daily maintenance: {daily_workflow_summary.get('duration_seconds', 0.0):.1f}s")

logger.info(f"\n🔧 Steps Executed: {steps_executed}")
logger.info(f"   Succeeded: {steps_succeeded}")
logger.info(f"   Failed: {steps_failed}")

logger.info(f"\n📚 Papers Processed:")
logger.info(f"   Weekly corpus discovery: {papers_discovered}")
logger.info(f"   Daily scan additions: {new_papers_daily}")
logger.info(f"   Total: {total_papers_processed}")

logger.info(f"\n🔬 Analysis Outputs:")
logger.info(f"   Citation network nodes: {network_nodes}")
logger.info(f"   Research gaps identified: {gaps_identified}")
logger.info(f"   Research questions updated: {rqs_updated}")

logger.info(f"\n📄 Reports Generated: {report_compilation_summary.get('reports_written', 0)}/5")

if all_failures:
    logger.info(f"\n⚠️  Failures: {len(all_failures)}")
    for failure in all_failures[:5]:  # Show first 5 failures
        logger.warning(f"   - {failure['step_name']}: {failure['error_message'][:80]}...")
    if len(all_failures) > 5:
        logger.warning(f"   ... and {len(all_failures) - 5} more (see manifest for full list)")

logger.info(f"\n📁 Output Directory: {orchestration_config.run_dir}")
logger.info(f"   Reports: {orchestration_config.reports_dir}")
logger.info(f"   Artifacts: {orchestration_config.artifacts_dir}")
logger.info(f"   Logs: {orchestration_config.logs_dir}")
logger.info(f"   Manifest: {manifest_path}")

if orchestration_config.dry_run:
    logger.warning(f"\n🔶 DRY RUN MODE - No writes to Notion/Drive were performed")

logger.info("\n" + "="*60)
logger.info(f"Run {orchestration_config.run_id} complete.")
logger.info("="*60)

# Store manifest data for potential further use
manifest_generation_duration = (datetime.now() - manifest_start_time).total_seconds()
logger.info(f"\nManifest generation completed in {manifest_generation_duration:.2f}s")

if VERBOSE:
    logger.debug("\nFull manifest structure:")
    logger.debug(json.dumps(run_manifest, indent=2, default=str))


2026-01-20 11:04:28 [INFO] orchestrator: 
2026-01-20 11:04:28 [INFO] orchestrator: GENERATING RUN MANIFEST
2026-01-20 11:04:28 [INFO] orchestrator: ============================================================
2026-01-20 11:04:28 [INFO] orchestrator: Aggregating step execution details...
2026-01-20 11:04:28 [INFO] orchestrator:   Captured details for 5 steps
2026-01-20 11:04:28 [INFO] orchestrator: Computing summary statistics...
2026-01-20 11:04:28 [INFO] orchestrator:   Summary statistics computed
2026-01-20 11:04:28 [INFO] orchestrator: Enumerating output files...
2026-01-20 11:04:28 [INFO] orchestrator:   Found 5 report files
2026-01-20 11:04:28 [INFO] orchestrator:   Found 0 checkpoint files
2026-01-20 11:04:28 [INFO] orchestrator:   Found 1 log files
2026-01-20 11:04:28 [INFO] orchestrator:   Found 0 artifact files
2026-01-20 11:04:28 [INFO] orchestrator: 
Writing manifest to run_manifest.json...
2026-01-20 11:04:28 [INFO] orchestrator:   ✅ Manifest written successfully (8,204 byt

In [33]:
# ============================================================
# Cell 15 — CLI and GitHub Actions orchestration skeleton
# ============================================================
# Overview:
#   Minimal CLI entry point and GitHub Actions workflow skeleton for
#   triggering pipeline runs. Supports command-line execution with flags
#   for dry-run, step selection, and notification configuration.
#
# Inputs / Outputs:
#   Inputs: Command-line arguments, environment variables
#   Outputs: Exit code (0=success, 1=failure), logged execution summary
#
# Notes:
#   - Implements argparse-based CLI for local execution
#   - Provides GitHub Actions workflow YAML template (as string constant)
#   - Supports selective step execution via --steps flag
#   - Integrates with failure notification system from cell 07
#   - Can be invoked as: python 025_notebook.py --dry-run --steps weekly

import argparse
import sys

# --- CLI argument parser ---
def create_cli_parser() -> argparse.ArgumentParser:
    """Create argument parser for CLI execution."""
    parser = argparse.ArgumentParser(
        description='Research automation pipeline orchestrator',
        formatter_class=argparse.RawDescriptionHelpFormatter
    )
    
    parser.add_argument('--dry-run', action='store_true',
                        help='Run in dry-run mode (no Notion/Drive writes)')
    parser.add_argument('--steps', choices=['weekly', 'analysis', 'daily', 'all'], default='all',
                        help='Select workflow to execute (default: all)')
    parser.add_argument('--no-reports', action='store_true',
                        help='Skip report generation')
    parser.add_argument('--verbose', action='store_true',
                        help='Enable verbose logging')
    
    return parser


# --- GitHub Actions workflow template ---
GITHUB_ACTIONS_WORKFLOW = '''
# .github/workflows/research_pipeline.yml
name: Research Pipeline

on:
  schedule:
    - cron: '0 2 * * 1'  # Weekly on Monday 2 AM UTC
  workflow_dispatch:  # Manual trigger
    inputs:
      dry_run:
        description: 'Run in dry-run mode'
        type: boolean
        default: false

jobs:
  run-pipeline:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - uses: actions/setup-python@v4
        with:
          python-version: '3.10'
      
      - name: Install dependencies
        run: |
          pip install -r requirements.txt
      
      - name: Run pipeline
        env:
          OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
          NOTION_TOKEN: ${{ secrets.NOTION_TOKEN }}
          LIT_DB: ${{ secrets.LIT_DB }}
          RQ_DB: ${{ secrets.RQ_DB }}
          MTG_DB: ${{ secrets.MTG_DB }}
        run: |
          python 025_notebook.py --steps all
      
      - name: Upload artifacts
        if: always()
        uses: actions/upload-artifact@v3
        with:
          name: pipeline-run
          path: runs/
'''


# --- CLI entry point ---
def cli_main():
    """Main CLI entry point."""
    parser = create_cli_parser()
    args = parser.parse_args()
    
    # Override global config flags from CLI
    global DRY_RUN, VERBOSE
    if args.dry_run:
        DRY_RUN = True
    if args.verbose:
        VERBOSE = True
        logger.setLevel(logging.DEBUG)
    
    # TODO: Apply step selection to orchestration_config
    # TODO: Execute run_pipeline() with filtered config
    # TODO: Return exit code based on success/failure
    
    logger.info(f"CLI invoked: steps={args.steps}, dry_run={args.dry_run}")
    logger.warning("[TODO] Full CLI integration not implemented - use cell 16 for execution")
    
    return 0


logger.info("✅ CLI and GitHub Actions skeleton defined")
logger.info("   GitHub Actions workflow template available in GITHUB_ACTIONS_WORKFLOW")
logger.info("   CLI entry: python 025_notebook.py --help")

if VERBOSE:
    logger.debug("GitHub Actions workflow:")
    logger.debug(GITHUB_ACTIONS_WORKFLOW)


2026-01-20 11:04:29 [INFO] orchestrator: ✅ CLI and GitHub Actions skeleton defined
2026-01-20 11:04:29 [INFO] orchestrator:    GitHub Actions workflow template available in GITHUB_ACTIONS_WORKFLOW
2026-01-20 11:04:29 [INFO] orchestrator:    CLI entry: python 025_notebook.py --help


In [34]:
# ============================================================
# Cell 16 — Run the full end-to-end pipeline with dry-run option
# ============================================================
# Overview:
#   Execute the complete end-to-end orchestration pipeline by calling
#   run_pipeline() with the configured OrchestrationConfig. Demonstrates
#   full integration of all seven steps (018-024) with comprehensive
#   error handling, reporting, and manifest generation. Can be toggled
#   between dry-run (testing) and production mode via DRY_RUN flag.
#
# Inputs / Outputs:
#   Inputs: orchestration_config (from cell 05), all global clients and configs
#   Outputs: Complete pipeline execution with all reports and manifest
#
# Notes:
#   - This is the main entry point for full pipeline execution
#   - Uses run_pipeline() orchestrator function from cell 09
#   - All prior cells (10-14) demonstrate individual workflows; this runs all
#   - Results are deterministically written to timestamped run directory
#   - Safe to re-run; each execution creates new isolated run directory
#   - Set DRY_RUN=true in env.txt to test without Notion/Drive writes
#

# --- Pre-execution validation ---
logger.info("\n" + "="*60)
logger.info("FULL PIPELINE EXECUTION")
logger.info("="*60)

logger.info("\nPre-execution validation...")

# Validate that all required clients are initialized
if openai_client is None:
    raise RuntimeError("OpenAI client not initialized. Run cells 01-04 first.")

if notion_client is None:
    raise RuntimeError("Notion client not initialized. Run cells 01-04 first.")

if orchestration_config is None:
    raise RuntimeError("Orchestration config not initialized. Run cell 05 first.")

logger.info("  ✅ All clients and configurations validated")

# Display execution mode
if orchestration_config.dry_run:
    logger.warning("\n🔶 RUNNING IN DRY-RUN MODE")
    logger.warning("   No writes to Notion or Google Drive will be performed")
    logger.warning("   All API calls will be simulated with placeholder data")
    logger.warning("   Set DRY_RUN=false in env.txt for production execution\n")
else:
    logger.info("\n✅ RUNNING IN PRODUCTION MODE")
    logger.info("   Writes to Notion and Google Drive ENABLED")
    logger.info("   All steps will execute with real API calls")
    logger.info("   Set DRY_RUN=true in env.txt to test safely\n")

# Confirm execution parameters
logger.info("Execution parameters:")
logger.info(f"  Run ID: {orchestration_config.run_id}")
logger.info(f"  Run timestamp: {orchestration_config.run_timestamp.strftime('%Y-%m-%d %H:%M:%S')}")
logger.info(f"  Output directory: {orchestration_config.run_dir}")
logger.info(f"  Max papers per run: {orchestration_config.max_papers_per_run}")
logger.info(f"  Timeout: {orchestration_config.timeout_seconds}s")
logger.info(f"  Max retries: {orchestration_config.max_retries}")

logger.info("\nEnabled workflow steps:")
step_flags = [
    ("Corpus discovery (018)", orchestration_config.enable_corpus_discovery),
    ("PDF extraction (019)", orchestration_config.enable_pdf_extraction),
    ("Notion upload (020)", orchestration_config.enable_notion_upload),
    ("Network analysis (021)", orchestration_config.enable_network_analysis),
    ("Gap analysis (022)", orchestration_config.enable_gap_analysis),
    ("Daily scan (023)", orchestration_config.enable_daily_scan),
    ("RQ update (024)", orchestration_config.enable_rq_update),
]

for step_name, enabled in step_flags:
    status = "✅ ENABLED" if enabled else "⏭️  DISABLED"
    logger.info(f"  {step_name}: {status}")

enabled_count = sum(1 for _, enabled in step_flags if enabled)
logger.info(f"\n  Total steps enabled: {enabled_count}/7")

if enabled_count == 0:
    logger.warning("\n⚠️  WARNING: No steps enabled. Enable at least one step in config.")
    logger.warning("   Execution will complete immediately with no operations performed.")

# --- Execute full pipeline ---
logger.info("\n" + "="*60)
logger.info("STARTING PIPELINE EXECUTION")
logger.info("="*60 + "\n")

pipeline_execution_start = datetime.now()

try:
    # Call main orchestrator function from cell 09
    pipeline_result = run_pipeline(orchestration_config)
    
    # Pipeline completed (may have failures but didn't crash)
    pipeline_execution_end = datetime.now()
    execution_wall_time = (pipeline_execution_end - pipeline_execution_start).total_seconds()
    
    logger.info("\n" + "="*60)
    logger.info("PIPELINE EXECUTION COMPLETED")
    logger.info("="*60)
    
    # Display high-level results
    logger.info(f"\nExecution wall time: {execution_wall_time:.1f}s ({execution_wall_time/60:.1f} minutes)")
    logger.info(f"Pipeline internal duration: {pipeline_result['duration_seconds']:.1f}s")
    
    if pipeline_result['success']:
        logger.info("\n✅ PIPELINE SUCCEEDED")
        logger.info("   All steps completed without failures")
    else:
        logger.warning("\n⚠️  PIPELINE COMPLETED WITH FAILURES")
        logger.warning(f"   {len(pipeline_result['failures'])} step(s) failed")
        logger.warning("   See run manifest for details")
    
    logger.info(f"\nSteps executed: {pipeline_result['steps_executed']}/7")
    
    # Show failure summary if any
    if pipeline_result['failures']:
        logger.warning("\nFailure summary:")
        for failure in pipeline_result['failures']:
            logger.warning(f"  ❌ {failure['step_name']}: {failure['error_message'][:100]}...")
    
    # Compile reports (already done in cell 13, but show summary here)
    logger.info("\n" + "="*60)
    logger.info("FINAL OUTPUTS")
    logger.info("="*60)
    
    logger.info(f"\n📁 Run directory: {orchestration_config.run_dir}")
    logger.info("\n📄 Generated reports:")
    
    for report_name, report_path in REPORT_FILES.items():
        if report_path.exists():
            size_kb = report_path.stat().st_size / 1024
            logger.info(f"   ✅ {report_path.name} ({size_kb:.1f} KB)")
        else:
            logger.warning(f"   ⚠️  {report_path.name} (not generated)")
    
    # Show manifest location
    manifest_path = orchestration_config.run_dir / 'run_manifest.json'
    if manifest_path.exists():
        manifest_size_kb = manifest_path.stat().st_size / 1024
        logger.info(f"\n📊 Run manifest: {manifest_path.name} ({manifest_size_kb:.1f} KB)")
        logger.info(f"   Full path: {manifest_path}")
    
    # Show checkpoint files
    checkpoint_count = sum(1 for cp in CHECKPOINT_FILES.values() if cp.exists())
    if checkpoint_count > 0:
        logger.info(f"\n💾 Checkpoint files: {checkpoint_count} saved")
        logger.info(f"   Location: {orchestration_config.artifacts_dir}")
    
    # Show log file
    log_file = orchestration_config.get_log_path('orchestrator.log')
    if log_file.exists():
        log_size_kb = log_file.stat().st_size / 1024
        logger.info(f"\n📝 Execution log: {log_file.name} ({log_size_kb:.1f} KB)")
    
    # Dry-run reminder
    if orchestration_config.dry_run:
        logger.warning("\n🔶 DRY-RUN MODE REMINDER")
        logger.warning("   This was a simulation - no actual writes to Notion/Drive")
        logger.warning("   Set DRY_RUN=false in env.txt for production execution")
    
    # Success banner
    logger.info("\n" + "="*60)
    if pipeline_result['success']:
        logger.info("🎉 PIPELINE EXECUTION SUCCESSFUL")
    else:
        logger.warning("⚠️  PIPELINE COMPLETED WITH ERRORS")
    logger.info(f"Run ID: {orchestration_config.run_id}")
    logger.info("="*60 + "\n")
    
    # Store result for notebook-level access
    final_pipeline_result = pipeline_result
    
except KeyboardInterrupt:
    logger.warning("\n⚠️  Pipeline execution interrupted by user (KeyboardInterrupt)")
    logger.warning("   Partial results may be available in checkpoint files")
    logger.warning(f"   Run directory: {orchestration_config.run_dir}")
    raise
    
except Exception as e:
    logger.error("\n❌ PIPELINE EXECUTION FAILED WITH EXCEPTION")
    logger.error(f"   Error: {str(e)}")
    logger.error(f"   Run directory: {orchestration_config.run_dir}")
    
    if VERBOSE:
        logger.error("\nFull traceback:")
        logger.error(traceback.format_exc())
    
    # Try to save error manifest
    try:
        error_manifest = {
            'run_id': orchestration_config.run_id,
            'status': 'FAILED',
            'error': str(e),
            'traceback': traceback.format_exc(),
            'timestamp': datetime.now().isoformat(),
        }
        error_manifest_path = orchestration_config.run_dir / 'error_manifest.json'
        error_manifest_path.write_text(json.dumps(error_manifest, indent=2))
        logger.info(f"\n💾 Error details saved to: {error_manifest_path}")
    except Exception as manifest_error:
        logger.error(f"Failed to save error manifest: {manifest_error}")
    
    # Re-raise for debugging
    raise

finally:
    # Cleanup and final logging
    logger.info("\nPipeline execution complete. Logs saved to:")
    logger.info(f"  {orchestration_config.get_log_path('orchestrator.log')}")
    logger.info(f"\nFor detailed results, see run manifest:")
    logger.info(f"  {orchestration_config.run_dir / 'run_manifest.json'}")
    logger.info("\n" + "="*60 + "\n")


2026-01-20 11:04:30 [INFO] orchestrator: 
2026-01-20 11:04:30 [INFO] orchestrator: FULL PIPELINE EXECUTION
2026-01-20 11:04:30 [INFO] orchestrator: ============================================================
2026-01-20 11:04:30 [INFO] orchestrator: 
Pre-execution validation...
2026-01-20 11:04:30 [INFO] orchestrator:   ✅ All clients and configurations validated
2026-01-20 11:04:30 [INFO] orchestrator: 
✅ RUNNING IN PRODUCTION MODE
2026-01-20 11:04:30 [INFO] orchestrator:    Writes to Notion and Google Drive ENABLED
2026-01-20 11:04:30 [INFO] orchestrator:    All steps will execute with real API calls
2026-01-20 11:04:30 [INFO] orchestrator:    Set DRY_RUN=true in env.txt to test safely

2026-01-20 11:04:30 [INFO] orchestrator: Execution parameters:
2026-01-20 11:04:30 [INFO] orchestrator:   Run ID: 20260120_104414
2026-01-20 11:04:30 [INFO] orchestrator:   Run timestamp: 2026-01-20 10:44:14
2026-01-20 11:04:30 [INFO] orchestrator:   Output directory: runs/2026-01-20_1044
2026-01-20 11